In [1]:
# ============================================
# BEDFORD 2023: MASTER POSSIBLE-EDGES TABLE
# Harvard-Oxford parcels -> Yeo-7 networks
# ============================================

import os
import re
import numpy as np
import pandas as pd
import nibabel as nib
from nilearn.image import resample_to_img

# =========================
# Paths
# =========================
BASE_DIR  = "/Users/maximegodart/Desktop/CBMA_coordinate_extract/CONNECTIVITY/NEW ANALYSIS DATA"
ATLAS_DIR = os.path.join(BASE_DIR, "Atlases")

HO_NII  = os.path.join(ATLAS_DIR, "conn_atlas_harvardoxford.nii")
HO_TXT  = os.path.join(ATLAS_DIR, "conn_atlas_harvardoxford.txt")
YEO_NII = os.path.join(ATLAS_DIR, "Yeo2011_7Networks_MNI152_FreeSurferConformed1mm.nii")

OUT_DIR = os.path.join(BASE_DIR, "possible_edge_outputs")
os.makedirs(OUT_DIR, exist_ok=True)

OUT_MASTER = os.path.join(OUT_DIR, "Bedford_2023_possible_edges_master.xlsx")

YEO7 = [
    "Visual",
    "Somatomotor",
    "DorsalAttention",
    "VentralAttention",
    "Limbic",
    "Frontoparietal",
    "Default",
]

# =========================
# Helpers
# =========================
def clean_ws(s):
    return re.sub(r"\s+", " ", str(s).strip())

def dice(a, b):
    inter = np.count_nonzero(a & b)
    na = np.count_nonzero(a)
    nb = np.count_nonzero(b)
    return 0.0 if (na + nb) == 0 else (2.0 * inter) / (na + nb)

def load_ho_label_to_id(txt_path):
    labels = []
    with open(txt_path, "r", encoding="utf-8", errors="ignore") as f:
        for ln in f:
            ln = ln.strip()
            if ln:
                labels.append(clean_ws(ln))
    return {lab: i + 1 for i, lab in enumerate(labels)}

def compute_ho_to_yeo7(ho_nii_path, yeo_nii_path):
    ho_img  = nib.load(ho_nii_path)
    yeo_img = nib.load(yeo_nii_path)

    yd = yeo_img.get_fdata()
    if yd.ndim == 4 and yd.shape[-1] == 1:
        yeo_img = nib.Nifti1Image(np.squeeze(yd, axis=-1), affine=yeo_img.affine)

    yeo_rs = resample_to_img(yeo_img, ho_img, interpolation="nearest")

    ho  = ho_img.get_fdata().astype(int)
    yeo = yeo_rs.get_fdata().astype(int)

    ho_ids  = [i for i in np.unique(ho) if i != 0]
    yeo_ids = sorted([i for i in np.unique(yeo) if i != 0])[:7]
    yeo_id_to_name = {yeo_ids[i]: YEO7[i] for i in range(min(7, len(yeo_ids)))}

    ho_to_yeo = {}
    ho_to_dice = {}

    for hid in ho_ids:
        hm = (ho == hid)
        best_name, best_d, best_inter = None, -1.0, -1

        for yid in yeo_ids:
            ym = (yeo == yid)
            inter = np.count_nonzero(hm & ym)
            d = dice(hm, ym)

            if (d > best_d) or (np.isclose(d, best_d) and inter > best_inter):
                best_d = float(d)
                best_inter = int(inter)
                best_name = yeo_id_to_name.get(yid)

        ho_to_yeo[hid] = best_name
        ho_to_dice[hid] = best_d

    return ho_to_yeo, ho_to_dice

def compute_yeo_counts_from_ho(ho_label_to_id, ho_to_yeo):
    rows = []
    for lab, hid in ho_label_to_id.items():
        rows.append({
            "HO_ID": hid,
            "HO_Label": lab,
            "Yeo7": ho_to_yeo.get(hid, None),
        })

    atlas_df = pd.DataFrame(rows).sort_values("HO_ID").reset_index(drop=True)
    counts = atlas_df.groupby("Yeo7")["HO_ID"].nunique().to_dict()
    counts = {k: int(counts.get(k, 0)) for k in YEO7}

    return atlas_df, counts

def possible_edges_for_pair(a, b, yeo_counts):
    nA = int(yeo_counts.get(a, 0))
    nB = int(yeo_counts.get(b, 0))

    if a == b:
        possible = nA * (nA - 1) // 2
        pair_type = "within"
    else:
        possible = nA * nB
        pair_type = "between"

    return nA, nB, possible, pair_type

# =========================
# Run
# =========================
ho_label_to_id = load_ho_label_to_id(HO_TXT)
ho_to_yeo, ho_to_dice = compute_ho_to_yeo7(HO_NII, YEO_NII)
atlas_df, yeo_counts = compute_yeo_counts_from_ho(ho_label_to_id, ho_to_yeo)

# Table 1: count how many HO parcels fall in each Yeo network
yeo_counts_df = pd.DataFrame({
    "Yeo7": YEO7,
    "N_HO_Parcels": [yeo_counts[k] for k in YEO7]
})

# Table 2: one row per Yeo network pair and its denominator
master_rows = []
for i, a in enumerate(YEO7):
    for b in YEO7[i:]:
        nA, nB, possible, pair_type = possible_edges_for_pair(a, b, yeo_counts)
        master_rows.append({
            "Study": "Bedford 2023",
            "Atlas": "Harvard-Oxford",
            "Yeo_Pair_A": a,
            "Yeo_Pair_B": b,
            "Regions_in_A": nA,
            "Regions_in_B": nB,
            "Possible_Edges": possible,
            "Pair_Type": pair_type,
        })

master_df = pd.DataFrame(master_rows)

# =========================
# Save
# =========================
with pd.ExcelWriter(OUT_MASTER, engine="openpyxl") as writer:
    yeo_counts_df.to_excel(writer, sheet_name="Yeo_counts", index=False)
    master_df.to_excel(writer, sheet_name="Master_possible_edges", index=False)
    atlas_df.to_excel(writer, sheet_name="HO_to_Yeo7_mapping", index=False)

print("Saved:", OUT_MASTER)
print("\nYeo counts:")
print(yeo_counts_df)

print("\nMaster possible-edge table:")
print(master_df.head(15))

Saved: /Users/maximegodart/Desktop/CBMA_coordinate_extract/CONNECTIVITY/NEW ANALYSIS DATA/possible_edge_outputs/Bedford_2023_possible_edges_master.xlsx

Yeo counts:
               Yeo7  N_HO_Parcels
0            Visual            55
1       Somatomotor            19
2   DorsalAttention             8
3  VentralAttention             8
4            Limbic            20
5    Frontoparietal            10
6           Default            12

Master possible-edge table:
           Study           Atlas       Yeo_Pair_A        Yeo_Pair_B  \
0   Bedford 2023  Harvard-Oxford           Visual            Visual   
1   Bedford 2023  Harvard-Oxford           Visual       Somatomotor   
2   Bedford 2023  Harvard-Oxford           Visual   DorsalAttention   
3   Bedford 2023  Harvard-Oxford           Visual  VentralAttention   
4   Bedford 2023  Harvard-Oxford           Visual            Limbic   
5   Bedford 2023  Harvard-Oxford           Visual    Frontoparietal   
6   Bedford 2023  Harvard-Oxford     

In [3]:
# ============================================
# DAI 2023: HCP ICA -> YEO-7 POSSIBLE EDGES
# Creates:
#   1) component-to-Yeo mapping from atlas overlap
#   2) Yeo counts based on original HCP ICA components
#   3) master table with one row per Yeo pair + denominator
# ============================================

import os
import re
import numpy as np
import pandas as pd
import nibabel as nib
from nilearn.image import resample_to_img

# =========================
# Paths
# =========================
BASE_DIR  = "/Users/maximegodart/Desktop/CBMA_coordinate_extract/CONNECTIVITY/NEW ANALYSIS DATA"
ATLAS_DIR = os.path.join(BASE_DIR, "Atlases")

DAI_XLSX = os.path.join(BASE_DIR, "Dai_2023_significant_edges_FINAL.xlsx")

HCPICA_PREFIX = "conn_networks_hcpica_32"
HCPICA_NII = None
for ext in (".nii.gz", ".nii"):
    p = os.path.join(ATLAS_DIR, HCPICA_PREFIX + ext)
    if os.path.exists(p):
        HCPICA_NII = p
        break
if HCPICA_NII is None:
    raise FileNotFoundError(f"Could not find {HCPICA_PREFIX}.nii(.gz) in: {ATLAS_DIR}")

HCPICA_TXT = os.path.join(ATLAS_DIR, "conn_networks_hcpica_32.txt")
if not os.path.exists(HCPICA_TXT):
    raise FileNotFoundError(f"Could not find {HCPICA_PREFIX}.txt in: {ATLAS_DIR}")

YEO_NII = os.path.join(ATLAS_DIR, "Yeo2011_7Networks_MNI152_FreeSurferConformed1mm.nii")
if not os.path.exists(YEO_NII):
    raise FileNotFoundError(f"Could not find Yeo atlas: {YEO_NII}")

OUT_DIR = os.path.join(BASE_DIR, "possible_edge_outputs")
os.makedirs(OUT_DIR, exist_ok=True)

OUT_XLSX = os.path.join(OUT_DIR, "Dai_2023_possible_edges_master.xlsx")

YEO7 = ["Visual", "Somatomotor", "DorsalAttention", "VentralAttention", "Limbic", "Frontoparietal", "Default"]

# =========================
# Helpers
# =========================
def clean_ws(s):
    return re.sub(r"\s+", " ", str(s).strip())

def dice(a, b):
    inter = np.count_nonzero(a & b)
    na = np.count_nonzero(a)
    nb = np.count_nonzero(b)
    return 0.0 if (na + nb) == 0 else (2.0 * inter) / (na + nb)

def dai_to_hcpica_base(s: str) -> str:
    """
    Convert Dai-style label to canonical HCP ICA base label.
    Examples:
      'Visual Medial' -> 'Visual.Medial'
      'Default Mode PCC' -> 'DefaultMode.PCC'
      'Language pSTG' -> 'Language.pSTG'
    """
    s0 = clean_ws(s)
    key = s0.lower()

    key = key.replace("default mode", "defaultmode")
    key = key.replace("dorsal attention", "dorsalattention")
    key = key.replace("frontoparietal", "frontoparietal")
    key = key.replace("sensorimotor", "sensorimotor")
    key = key.replace("salience", "salience")
    key = key.replace("language", "language")
    key = key.replace("visual", "visual")
    key = key.replace("cerebellar", "cerebellar")

    net_case = {
        "defaultmode": "DefaultMode",
        "dorsalattention": "DorsalAttention",
        "frontoparietal": "FrontoParietal",
        "sensorimotor": "SensoriMotor",
        "salience": "Salience",
        "language": "Language",
        "visual": "Visual",
        "cerebellar": "Cerebellar",
    }

    parts = key.split(" ", 1)
    if len(parts) != 2 or parts[0] not in net_case:
        raise ValueError(f"Unrecognized Dai HCP-ICA label: {s!r}")

    net = net_case[parts[0]]
    node_raw = parts[1].strip()

    node_case = {
        "medial": "Medial",
        "lateral": "Lateral",
        "occipital": "Occipital",
        "superior": "Superior",
        "mpfc": "MPFC",
        "pcc": "PCC",
        "lp": "LP",
        "fef": "FEF",
        "ips": "IPS",
        "ifg": "IFG",
        "pstg": "pSTG",
        "ainsula": "AInsula",
        "rpfc": "RPFC",
        "smg": "SMG",
        "lpfc": "LPFC",
        "ppc": "PPC",
        "anterior": "Anterior",
        "posterior": "Posterior",
    }

    node_key = node_raw.replace(" ", "")
    node = node_case.get(node_key) or node_case.get(node_raw)
    if node is None:
        raise ValueError(f"Unrecognized Dai HCP-ICA node/subregion: {s!r} -> {node_key!r}")

    return f"{net}.{node}"

def load_hcpica_txt_labels(txt_path: str):
    """
    Parse conn_networks_hcpica_32.txt and return:
      base_label -> list of component indices (0-based)
    stripping coordinate tuples and hemisphere tags.
    """
    base_to_idxs = {}
    names = []

    with open(txt_path, "r", encoding="utf-8", errors="ignore") as f:
        for ln in f:
            ln = ln.strip()
            if ln:
                names.append(ln)

    for i, name in enumerate(names):
        name_no_xyz = re.sub(r"\s*\(\s*-?\d+\s*,\s*-?\d+\s*,\s*-?\d+\s*\)\s*$", "", name).strip()
        base = re.sub(r"\s*\([LR]\)\s*$", "", name_no_xyz).strip()
        base_to_idxs.setdefault(base, []).append(i)

    return base_to_idxs

def compute_hcpica_to_yeo7(hcpica_img, yeo_img, base_to_idxs, n_comps_expected=32):
    """
    For each HCP ICA base label, union L/R components if present
    and assign the Yeo-7 network with maximum Dice overlap.
    """
    hcp = hcpica_img.get_fdata()
    if hcp.ndim != 4:
        raise ValueError(f"Expected HCPICA NIfTI to be 4D. Got shape: {hcp.shape}")
    if n_comps_expected is not None and hcp.shape[-1] < n_comps_expected:
        raise ValueError(f"Expected >= {n_comps_expected} components. Got {hcp.shape[-1]}")

    yd = yeo_img.get_fdata()
    if yd.ndim == 4 and yd.shape[-1] == 1:
        yeo_img = nib.Nifti1Image(np.squeeze(yd, axis=-1), affine=yeo_img.affine)

    yeo_rs = resample_to_img(yeo_img, hcpica_img, interpolation="nearest")
    yeo = yeo_rs.get_fdata().astype(int)

    yeo_ids = sorted([i for i in np.unique(yeo) if i != 0])[:7]
    yeo_id_to_name = {yeo_ids[i]: YEO7[i] for i in range(min(7, len(yeo_ids)))}
    yeo_masks = {yid: (yeo == yid) for yid in yeo_ids}

    def comp_mask(k):
        return np.abs(hcp[..., k]) > 0

    base_to_yeo = {}
    base_to_dice = {}

    for base, idxs in base_to_idxs.items():
        hm = np.zeros(yeo.shape, dtype=bool)
        for k in idxs:
            hm |= comp_mask(k)

        best_name, best_d, best_inter = None, -1.0, -1
        for yid, ym in yeo_masks.items():
            inter = np.count_nonzero(hm & ym)
            d = dice(hm, ym)
            if (d > best_d) or (np.isclose(d, best_d) and inter > best_inter):
                best_d, best_inter = float(d), int(inter)
                best_name = yeo_id_to_name.get(yid)

        base_to_yeo[base] = best_name
        base_to_dice[base] = best_d

    return base_to_yeo, base_to_dice

def require_map(base, base_to_yeo, row_idx, colname):
    if base not in base_to_yeo:
        alt = next((k for k in base_to_yeo.keys() if k.lower() == base.lower()), None)
        if alt is not None:
            base = alt
        else:
            hints = [k for k in sorted(base_to_yeo.keys()) if base.split(".")[0].lower() in k.lower()][:10]
            hint_txt = ("\nClosest (same network):\n  - " + "\n  - ".join(hints)) if hints else ""
            raise ValueError(
                f"[Mapping error] Row {row_idx}: '{colname}' base label not found.\n"
                f"  Canonical: {base!r}{hint_txt}"
            )

    yeo = base_to_yeo.get(base)
    if yeo not in YEO7:
        raise ValueError(
            f"[Mapping error] Row {row_idx}: '{colname}' got invalid Yeo-7 value: {yeo!r}\n"
            f"  Canonical: {base!r}"
        )
    return yeo

def possible_edges_for_pair(a, b, yeo_counts):
    nA = int(yeo_counts.get(a, 0))
    nB = int(yeo_counts.get(b, 0))
    if a == b:
        return nA, nB, nA * (nA - 1) // 2, "within"
    return nA, nB, nA * nB, "between"

# =========================
# Build HCP ICA -> Yeo-7 mapping from atlas overlap
# =========================
hcpica_img = nib.load(HCPICA_NII)
yeo_img = nib.load(YEO_NII)

base_to_idxs = load_hcpica_txt_labels(HCPICA_TXT)
base_to_yeo, base_to_dice = compute_hcpica_to_yeo7(hcpica_img, yeo_img, base_to_idxs, n_comps_expected=32)

# =========================
# Build component/base mapping table and Yeo counts
# =========================
mapping_rows = []
for base in sorted(base_to_idxs.keys()):
    mapping_rows.append({
        "HCPICA_Base": base,
        "N_Components_Unioned": len(base_to_idxs[base]),
        "Yeo7": base_to_yeo.get(base),
        "Dice": base_to_dice.get(base),
    })
mapping_df = pd.DataFrame(mapping_rows)

yeo_counts = (
    mapping_df.groupby("Yeo7", as_index=False)["HCPICA_Base"]
    .nunique()
    .rename(columns={"HCPICA_Base": "N_HCPICA_Bases"})
)
yeo_counts = yeo_counts.set_index("Yeo7")["N_HCPICA_Bases"].to_dict()
yeo_counts = {k: int(yeo_counts.get(k, 0)) for k in YEO7}

yeo_counts_df = pd.DataFrame({
    "Yeo7": YEO7,
    "N_HCPICA_Bases": [yeo_counts[k] for k in YEO7]
})

# =========================
# Master table: one row per Yeo pair and denominator
# =========================
master_rows = []
for i, a in enumerate(YEO7):
    for b in YEO7[i:]:
        nA, nB, possible, pair_type = possible_edges_for_pair(a, b, yeo_counts)
        master_rows.append({
            "Study": "Dai 2023",
            "Atlas": "HCP ICA",
            "Yeo_Pair_A": a,
            "Yeo_Pair_B": b,
            "Regions_in_A": nA,
            "Regions_in_B": nB,
            "Possible_Edges": possible,
            "Pair_Type": pair_type,
        })
master_df = pd.DataFrame(master_rows)

# =========================
# Optional: row-level table for reported Dai edges
# =========================
df = pd.read_excel(DAI_XLSX)
df.columns = df.columns.astype(str).str.replace("\u00a0", " ", regex=False).str.strip()

seed_c, targ_c, dir_c, net_c, con_c = "Seed", "Target", "Direction", "Network", "Contrast"

net_norm = (
    df[net_c].astype(str).str.strip().str.lower()
      .str.replace(r"[^a-z0-9]+", "", regex=True)
)
sub = df[net_norm.eq("hcpica")].copy()

seed_base = []
targ_base = []
seed_yeo = []
targ_yeo = []
pair_a = []
pair_b = []
possible_list = []
pair_type_list = []

for i, r in sub.iterrows():
    s_base = dai_to_hcpica_base(r[seed_c])
    t_base = dai_to_hcpica_base(r[targ_c])
    s_yeo = require_map(s_base, base_to_yeo, i, "Seed")
    t_yeo = require_map(t_base, base_to_yeo, i, "Target")

    A, B = (s_yeo, t_yeo) if YEO7.index(s_yeo) <= YEO7.index(t_yeo) else (t_yeo, s_yeo)
    _, _, possible, pair_type = possible_edges_for_pair(A, B, yeo_counts)

    seed_base.append(s_base)
    targ_base.append(t_base)
    seed_yeo.append(s_yeo)
    targ_yeo.append(t_yeo)
    pair_a.append(A)
    pair_b.append(B)
    possible_list.append(possible)
    pair_type_list.append(pair_type)

rowlevel_df = pd.DataFrame({
    "Study": "Dai",
    "Contrast": sub[con_c].astype(str).values,
    "Seed_Original": sub[seed_c].astype(str).values,
    "Target_Original": sub[targ_c].astype(str).values,
    "Seed_HCPICA_Base": seed_base,
    "Target_HCPICA_Base": targ_base,
    "Seed_Yeo7": seed_yeo,
    "Target_Yeo7": targ_yeo,
    "Yeo_Pair_A": pair_a,
    "Yeo_Pair_B": pair_b,
    "Direction": sub[dir_c].astype(str).values,
    "Possible_Edges": possible_list,
    "Pair_Type": pair_type_list,
})

# =========================
# Save
# =========================
with pd.ExcelWriter(OUT_XLSX, engine="openpyxl") as writer:
    mapping_df.to_excel(writer, sheet_name="HCPICA_to_Yeo7_mapping", index=False)
    yeo_counts_df.to_excel(writer, sheet_name="Yeo_counts", index=False)
    master_df.to_excel(writer, sheet_name="Master_possible_edges", index=False)
    rowlevel_df.to_excel(writer, sheet_name="Rowlevel_denominators", index=False)

print("✅ Saved:", OUT_XLSX)
print("HCPICA NIfTI used:", HCPICA_NII)
print("\nYeo counts:")
print(yeo_counts_df)
print("\nMaster possible-edge table:")
print(master_df.head(15))

✅ Saved: /Users/maximegodart/Desktop/CBMA_coordinate_extract/CONNECTIVITY/NEW ANALYSIS DATA/possible_edge_outputs/Dai_2023_possible_edges_master.xlsx
HCPICA NIfTI used: /Users/maximegodart/Desktop/CBMA_coordinate_extract/CONNECTIVITY/NEW ANALYSIS DATA/Atlases/conn_networks_hcpica_32.nii

Yeo counts:
               Yeo7  N_HCPICA_Bases
0            Visual               5
1       Somatomotor               3
2   DorsalAttention               1
3  VentralAttention               4
4            Limbic               0
5    Frontoparietal               2
6           Default               5

Master possible-edge table:
       Study    Atlas       Yeo_Pair_A        Yeo_Pair_B  Regions_in_A  \
0   Dai 2023  HCP ICA           Visual            Visual             5   
1   Dai 2023  HCP ICA           Visual       Somatomotor             5   
2   Dai 2023  HCP ICA           Visual   DorsalAttention             5   
3   Dai 2023  HCP ICA           Visual  VentralAttention             5   
4   Dai 2023

In [4]:
# ============================================
# DE ARAUJO 2011: BRODMANN -> YEO-7 POSSIBLE EDGES
# Creates:
#   1) BA-to-Yeo mapping from atlas overlap
#   2) Yeo counts based on original Brodmann areas
#   3) master table with one row per Yeo pair + denominator
# ============================================

import os
import re
import numpy as np
import pandas as pd
import nibabel as nib
from nilearn.image import resample_to_img

# =========================
# Paths
# =========================
BASE_DIR  = "/Users/maximegodart/Desktop/CBMA_coordinate_extract/CONNECTIVITY/NEW ANALYSIS DATA"
ATLAS_DIR = os.path.join(BASE_DIR, "Atlases")

DEARAUJO_XLSX = os.path.join(BASE_DIR, "DeAraujo_2011_significant_edges_FINAL.xlsx")
BROD_NII      = os.path.join(ATLAS_DIR, "brodmann.nii")
YEO_NII       = os.path.join(ATLAS_DIR, "Yeo2011_7Networks_MNI152_FreeSurferConformed1mm.nii")

if not os.path.exists(BROD_NII):
    raise FileNotFoundError(f"Could not find Brodmann atlas: {BROD_NII}")
if not os.path.exists(YEO_NII):
    raise FileNotFoundError(f"Could not find Yeo atlas: {YEO_NII}")

OUT_DIR = os.path.join(BASE_DIR, "possible_edge_outputs")
os.makedirs(OUT_DIR, exist_ok=True)

OUT_XLSX = os.path.join(OUT_DIR, "DeAraujo_2011_possible_edges_master.xlsx")

YEO7 = ["Visual","Somatomotor","DorsalAttention","VentralAttention","Limbic","Frontoparietal","Default"]

# =========================
# Helpers
# =========================
def clean_ws(s):
    return re.sub(r"\s+", " ", str(s).strip())

def dice(a, b):
    inter = np.count_nonzero(a & b)
    na = np.count_nonzero(a)
    nb = np.count_nonzero(b)
    return 0.0 if (na + nb) == 0 else (2.0 * inter) / (na + nb)

def parse_ba(x):
    """
    Accepts labels like: 'BA10', 'BA 10', 'ba 10'
    Returns integer BA id.
    """
    s = clean_ws(x).lower()
    m = re.match(r"^ba\s*0*(\d+)$", s)
    if not m:
        raise ValueError(f"Unrecognized Brodmann label: {x!r}")
    return int(m.group(1))

def compute_ba_to_yeo7(brod_img, yeo_img):
    """
    For each nonzero BA id in brodmann.nii,
    assign the Yeo-7 label with maximum Dice overlap.
    """
    brod = brod_img.get_fdata().astype(int)
    if brod.ndim != 3:
        raise ValueError(f"Expected Brodmann atlas to be 3D. Got shape: {brod.shape}")

    yd = yeo_img.get_fdata()
    if yd.ndim == 4 and yd.shape[-1] == 1:
        yeo_img = nib.Nifti1Image(np.squeeze(yd, axis=-1), affine=yeo_img.affine)

    yeo_rs = resample_to_img(yeo_img, brod_img, interpolation="nearest")
    yeo = yeo_rs.get_fdata().astype(int)

    ba_ids  = [i for i in np.unique(brod) if i != 0]
    yeo_ids = sorted([i for i in np.unique(yeo) if i != 0])[:7]
    yeo_id_to_name = {yeo_ids[i]: YEO7[i] for i in range(min(7, len(yeo_ids)))}

    ba_to_yeo = {}
    ba_to_dice = {}

    for bid in ba_ids:
        bm = (brod == bid)
        best_name, best_d, best_inter = None, -1.0, -1

        for yid in yeo_ids:
            ym = (yeo == yid)
            inter = np.count_nonzero(bm & ym)
            d = dice(bm, ym)
            if (d > best_d) or (np.isclose(d, best_d) and inter > best_inter):
                best_d, best_inter = float(d), int(inter)
                best_name = yeo_id_to_name.get(yid)

        ba_to_yeo[bid] = best_name
        ba_to_dice[bid] = best_d

    return ba_to_yeo, ba_to_dice

def require_map_ba(label, ba_to_yeo, row_idx, colname):
    raw = label
    bid = parse_ba(raw)

    if bid not in ba_to_yeo:
        known = sorted(ba_to_yeo.keys())
        raise ValueError(
            f"[Mapping error] Row {row_idx}: '{colname}' BA id {bid} not found in brodmann.nii.\n"
            f"  Raw value: {raw!r}\n"
            f"  Known BA ids (sample): {known[:25]}{' ...' if len(known) > 25 else ''}"
        )

    yeo = ba_to_yeo.get(bid)
    if yeo not in YEO7:
        raise ValueError(
            f"[Mapping error] Row {row_idx}: '{colname}' BA{bid} mapped to invalid Yeo-7 value: {yeo!r}\n"
            f"  Raw value: {raw!r}"
        )

    return bid, yeo

def possible_edges_for_pair(a, b, yeo_counts):
    nA = int(yeo_counts.get(a, 0))
    nB = int(yeo_counts.get(b, 0))
    if a == b:
        return nA, nB, nA * (nA - 1) // 2, "within"
    return nA, nB, nA * nB, "between"

# =========================
# Build BA -> Yeo-7 mapping
# =========================
brod_img = nib.load(BROD_NII)
yeo_img  = nib.load(YEO_NII)

ba_to_yeo, ba_to_dice = compute_ba_to_yeo7(brod_img, yeo_img)

# =========================
# Build mapping table and Yeo counts
# =========================
mapping_rows = []
for bid in sorted(ba_to_yeo.keys()):
    mapping_rows.append({
        "BA_ID": bid,
        "BA_Label": f"BA{bid}",
        "Yeo7": ba_to_yeo.get(bid),
        "Dice": ba_to_dice.get(bid),
    })
mapping_df = pd.DataFrame(mapping_rows)

yeo_counts = (
    mapping_df.groupby("Yeo7", as_index=False)["BA_ID"]
    .nunique()
    .rename(columns={"BA_ID": "N_Brodmann_Areas"})
)
yeo_counts = yeo_counts.set_index("Yeo7")["N_Brodmann_Areas"].to_dict()
yeo_counts = {k: int(yeo_counts.get(k, 0)) for k in YEO7}

yeo_counts_df = pd.DataFrame({
    "Yeo7": YEO7,
    "N_Brodmann_Areas": [yeo_counts[k] for k in YEO7]
})

# =========================
# Master table: one row per Yeo pair
# =========================
master_rows = []
for i, a in enumerate(YEO7):
    for b in YEO7[i:]:
        nA, nB, possible, pair_type = possible_edges_for_pair(a, b, yeo_counts)
        master_rows.append({
            "Study": "de Araujo 2011",
            "Atlas": "Brodmann",
            "Yeo_Pair_A": a,
            "Yeo_Pair_B": b,
            "Regions_in_A": nA,
            "Regions_in_B": nB,
            "Possible_Edges": possible,
            "Pair_Type": pair_type,
        })
master_df = pd.DataFrame(master_rows)

# =========================
# Optional: row-level table for reported edges
# =========================
df = pd.read_excel(DEARAUJO_XLSX)
df.columns = df.columns.astype(str).str.replace("\u00a0", " ", regex=False).str.strip()

seed_c, targ_c, dir_c, net_c, con_c = "Seed", "Target", "Direction", "Network", "Contrast"

net_norm = (
    df[net_c].astype(str).str.strip().str.lower()
      .str.replace(r"[^a-z0-9]+", "", regex=True)
)
sub = df[net_norm.isin(["brodmannarea", "brodmann", "ba"])].copy()

seed_ba = []
targ_ba = []
seed_yeo = []
targ_yeo = []
pair_a = []
pair_b = []
possible_list = []
pair_type_list = []

for i, r in sub.iterrows():
    s_bid, s_yeo = require_map_ba(r[seed_c], ba_to_yeo, i, "Seed")
    t_bid, t_yeo = require_map_ba(r[targ_c], ba_to_yeo, i, "Target")

    A, B = (s_yeo, t_yeo) if YEO7.index(s_yeo) <= YEO7.index(t_yeo) else (t_yeo, s_yeo)
    _, _, possible, pair_type = possible_edges_for_pair(A, B, yeo_counts)

    seed_ba.append(f"BA{s_bid}")
    targ_ba.append(f"BA{t_bid}")
    seed_yeo.append(s_yeo)
    targ_yeo.append(t_yeo)
    pair_a.append(A)
    pair_b.append(B)
    possible_list.append(possible)
    pair_type_list.append(pair_type)

rowlevel_df = pd.DataFrame({
    "Study": "de Araujo",
    "Contrast": sub[con_c].astype(str).values,
    "Seed_Original": sub[seed_c].astype(str).values,
    "Target_Original": sub[targ_c].astype(str).values,
    "Seed_BA": seed_ba,
    "Target_BA": targ_ba,
    "Seed_Yeo7": seed_yeo,
    "Target_Yeo7": targ_yeo,
    "Yeo_Pair_A": pair_a,
    "Yeo_Pair_B": pair_b,
    "Direction": sub[dir_c].astype(str).values,
    "Possible_Edges": possible_list,
    "Pair_Type": pair_type_list,
})

# =========================
# Save
# =========================
with pd.ExcelWriter(OUT_XLSX, engine="openpyxl") as writer:
    mapping_df.to_excel(writer, sheet_name="BA_to_Yeo7_mapping", index=False)
    yeo_counts_df.to_excel(writer, sheet_name="Yeo_counts", index=False)
    master_df.to_excel(writer, sheet_name="Master_possible_edges", index=False)
    rowlevel_df.to_excel(writer, sheet_name="Rowlevel_denominators", index=False)

print("✅ Saved:", OUT_XLSX)
print("Brodmann atlas used:", BROD_NII)
print("\nYeo counts:")
print(yeo_counts_df)
print("\nMaster possible-edge table:")
print(master_df.head(15))

✅ Saved: /Users/maximegodart/Desktop/CBMA_coordinate_extract/CONNECTIVITY/NEW ANALYSIS DATA/possible_edge_outputs/DeAraujo_2011_possible_edges_master.xlsx
Brodmann atlas used: /Users/maximegodart/Desktop/CBMA_coordinate_extract/CONNECTIVITY/NEW ANALYSIS DATA/Atlases/brodmann.nii

Yeo counts:
               Yeo7  N_Brodmann_Areas
0            Visual                 5
1       Somatomotor                 6
2   DorsalAttention                 5
3  VentralAttention                 3
4            Limbic                 8
5    Frontoparietal                 6
6           Default                 8

Master possible-edge table:
             Study     Atlas       Yeo_Pair_A        Yeo_Pair_B  Regions_in_A  \
0   de Araujo 2011  Brodmann           Visual            Visual             5   
1   de Araujo 2011  Brodmann           Visual       Somatomotor             5   
2   de Araujo 2011  Brodmann           Visual   DorsalAttention             5   
3   de Araujo 2011  Brodmann           Visual  Ven

In [5]:
# ============================================
# GRIMM 2018: AAL SEED + MNI TARGET SPHERES -> YEO-7 POSSIBLE EDGES
# Creates:
#   1) AAL ROI-to-Yeo mapping from atlas overlap
#   2) target-sphere-to-Yeo assignments for reported targets
#   3) Yeo counts based on original AAL regions
#   4) master table with one row per Yeo pair + denominator
# ============================================

import os
import re
import numpy as np
import pandas as pd
import nibabel as nib
from nilearn.image import resample_to_img

# =========================
# Paths
# =========================
BASE_DIR  = "/Users/maximegodart/Desktop/CBMA_coordinate_extract/CONNECTIVITY/NEW ANALYSIS DATA"
ATLAS_DIR = os.path.join(BASE_DIR, "Atlases")

GRIMM_XLSX = os.path.join(BASE_DIR, "Grimm_2018_significant edges_FINAL.xlsx")

AAL_NII = os.path.join(ATLAS_DIR, "AAL3v1_1mm.nii.gz")
AAL_TXT = os.path.join(ATLAS_DIR, "AAL3v1_1mm.nii.txt")
YEO_NII = os.path.join(ATLAS_DIR, "Yeo2011_7Networks_MNI152_FreeSurferConformed1mm.nii")

if not os.path.exists(AAL_NII):
    raise FileNotFoundError(f"Could not find AAL atlas: {AAL_NII}")
if not os.path.exists(AAL_TXT):
    raise FileNotFoundError(f"Could not find AAL label file: {AAL_TXT}")
if not os.path.exists(YEO_NII):
    raise FileNotFoundError(f"Could not find Yeo atlas: {YEO_NII}")

OUT_DIR = os.path.join(BASE_DIR, "possible_edge_outputs")
os.makedirs(OUT_DIR, exist_ok=True)

OUT_XLSX = os.path.join(OUT_DIR, "Grimm_2018_possible_edges_master.xlsx")

YEO7 = ["Visual","Somatomotor","DorsalAttention","VentralAttention","Limbic","Frontoparietal","Default"]

# =========================
# Helpers
# =========================
def clean_ws(s):
    return re.sub(r"\s+", " ", str(s).strip())

def dice(a, b):
    inter = np.count_nonzero(a & b)
    na = np.count_nonzero(a)
    nb = np.count_nonzero(b)
    return 0.0 if (na + nb) == 0 else (2.0 * inter) / (na + nb)

def sphere_mask_on_grid(ref_img: nib.Nifti1Image, center_xyz_mm, radius_mm=10.0):
    """Boolean sphere mask in voxel grid of ref_img."""
    shape = ref_img.shape[:3]
    aff = ref_img.affine
    inv = np.linalg.inv(aff)

    cx, cy, cz = nib.affines.apply_affine(inv, np.array(center_xyz_mm, float))
    cx, cy, cz = float(cx), float(cy), float(cz)

    vx = float(np.sqrt((aff[:3, 0] ** 2).sum()))
    vy = float(np.sqrt((aff[:3, 1] ** 2).sum()))
    vz = float(np.sqrt((aff[:3, 2] ** 2).sum()))

    rx = int(np.ceil(radius_mm / vx))
    ry = int(np.ceil(radius_mm / vy))
    rz = int(np.ceil(radius_mm / vz))

    x0 = max(0, int(np.floor(cx)) - rx); x1 = min(shape[0], int(np.floor(cx)) + rx + 1)
    y0 = max(0, int(np.floor(cy)) - ry); y1 = min(shape[1], int(np.floor(cy)) + ry + 1)
    z0 = max(0, int(np.floor(cz)) - rz); z1 = min(shape[2], int(np.floor(cz)) + rz + 1)

    xs = np.arange(x0, x1)
    ys = np.arange(y0, y1)
    zs = np.arange(z0, z1)
    X, Y, Z = np.meshgrid(xs, ys, zs, indexing="ij")

    dx = (X - cx) * vx
    dy = (Y - cy) * vy
    dz = (Z - cz) * vz
    local = (dx * dx + dy * dy + dz * dz) <= (radius_mm ** 2)

    m = np.zeros(shape, dtype=bool)
    m[x0:x1, y0:y1, z0:z1] = local
    return m

def yeo_masks_resampled_to(ref_img: nib.Nifti1Image, yeo_path: str):
    """Resample Yeo-7 to ref_img grid and return dict: yeo_name -> bool mask."""
    yeo_img = nib.load(yeo_path)
    yd = yeo_img.get_fdata()
    if yd.ndim == 4 and yd.shape[-1] == 1:
        yeo_img = nib.Nifti1Image(np.squeeze(yd, axis=-1), affine=yeo_img.affine)

    yeo_rs = resample_to_img(yeo_img, ref_img, interpolation="nearest")
    yeo = yeo_rs.get_fdata().astype(int)

    yeo_ids = sorted([i for i in np.unique(yeo) if i != 0])[:7]
    yeo_id_to_name = {yeo_ids[i]: YEO7[i] for i in range(min(7, len(yeo_ids)))}

    masks = {}
    for yid in yeo_ids:
        nm = yeo_id_to_name.get(yid)
        if nm:
            masks[nm] = (yeo == yid)
    return masks

def best_yeo_for_mask(mask: np.ndarray, yeo_masks: dict):
    """Return Yeo-7 name with max Dice overlap; tie-break by intersection."""
    best_name, best_d, best_inter = None, -1.0, -1
    for nm, ym in yeo_masks.items():
        inter = np.count_nonzero(mask & ym)
        d = dice(mask, ym)
        if (d > best_d) or (np.isclose(d, best_d) and inter > best_inter):
            best_d, best_inter = float(d), int(inter)
            best_name = nm
    return best_name, best_d

def load_aal_label_to_id(aal_txt_path: str):
    """
    Expect one label per line or 'ID LABEL' format.
    Returns dict: label -> id (1..N)
    """
    labels = []
    with open(aal_txt_path, "r", encoding="utf-8", errors="ignore") as f:
        for ln in f:
            ln = ln.strip()
            if not ln:
                continue
            m = re.match(r"^(\d+)\s+(.+)$", ln)
            if m:
                labels.append(clean_ws(m.group(2)))
            else:
                labels.append(clean_ws(ln))
    return {lab: i + 1 for i, lab in enumerate(labels)}

def load_aal_id_to_label(aal_txt_path: str):
    label_to_id = load_aal_label_to_id(aal_txt_path)
    return {v: k for k, v in label_to_id.items()}

def compute_aal_to_yeo(aal_img, yeo_masks, aal_ids):
    aal = aal_img.get_fdata().astype(int)
    aal_to_yeo = {}
    aal_to_dice = {}
    for aid in aal_ids:
        mask = (aal == aid)
        nm, d = best_yeo_for_mask(mask, yeo_masks)
        aal_to_yeo[aid] = nm
        aal_to_dice[aid] = d
    return aal_to_yeo, aal_to_dice

def require_aal_mask(aal_img, aal_label_to_id, seed_label_raw, row_idx):
    """
    Convert Grimm seed shorthand -> exact AAL label -> boolean mask.
    Update SEED_ALIASES if your exact AAL file uses different naming.
    """
    SEED_ALIASES = {
        "r amygdala": [
            "Amygdala_R",
            "Amygdala_R (Amygdala Right)",
            "Amygdala R",
            "Right Amygdala",
            "Amygdala_R (Amygdala Right Hemisphere)",
        ]
    }

    key = clean_ws(seed_label_raw).lower()
    candidates = SEED_ALIASES.get(key, [seed_label_raw])

    found = None
    for cand in candidates:
        cand_clean = clean_ws(cand)
        if cand_clean in aal_label_to_id:
            found = cand_clean
            break

    if found is None:
        hints = [k for k in aal_label_to_id.keys()
                 if ("amygdala" in k.lower()) or ("amyg" in k.lower())][:20]
        raise ValueError(
            f"[Seed mapping error] Row {row_idx}: seed {seed_label_raw!r} not found in AAL labels.\n"
            f"Try updating SEED_ALIASES with the exact AAL label string.\n"
            f"Example AAL 'amygdala' labels:\n  - " + "\n  - ".join(hints)
        )

    sid = aal_label_to_id[found]
    aal = aal_img.get_fdata().astype(int)
    return sid, found, (aal == sid)

def possible_edges_for_pair(a, b, yeo_counts):
    nA = int(yeo_counts.get(a, 0))
    nB = int(yeo_counts.get(b, 0))
    if a == b:
        return nA, nB, nA * (nA - 1) // 2, "within"
    return nA, nB, nA * nB, "between"

# =========================
# Load atlas data
# =========================
aal_img = nib.load(AAL_NII)
aal_label_to_id = load_aal_label_to_id(AAL_TXT)
aal_id_to_label = load_aal_id_to_label(AAL_TXT)
aal = aal_img.get_fdata().astype(int)

yeo_masks = yeo_masks_resampled_to(aal_img, YEO_NII)

# All original AAL ids present in atlas
aal_ids = [i for i in np.unique(aal) if i != 0]

# AAL ROI -> Yeo mapping from atlas overlap
aal_to_yeo, aal_to_dice = compute_aal_to_yeo(aal_img, yeo_masks, aal_ids)

# =========================
# Mapping table and Yeo counts from original AAL atlas
# =========================
mapping_rows = []
for aid in sorted(aal_ids):
    mapping_rows.append({
        "AAL_ID": aid,
        "AAL_Label": aal_id_to_label.get(aid, f"AAL_{aid}"),
        "Yeo7": aal_to_yeo.get(aid),
        "Dice": aal_to_dice.get(aid),
    })
mapping_df = pd.DataFrame(mapping_rows)

yeo_counts = (
    mapping_df.groupby("Yeo7", as_index=False)["AAL_ID"]
    .nunique()
    .rename(columns={"AAL_ID": "N_AAL_Regions"})
)
yeo_counts = yeo_counts.set_index("Yeo7")["N_AAL_Regions"].to_dict()
yeo_counts = {k: int(yeo_counts.get(k, 0)) for k in YEO7}

yeo_counts_df = pd.DataFrame({
    "Yeo7": YEO7,
    "N_AAL_Regions": [yeo_counts[k] for k in YEO7]
})

# =========================
# Master table: one row per Yeo pair
# =========================
master_rows = []
for i, a in enumerate(YEO7):
    for b in YEO7[i:]:
        nA, nB, possible, pair_type = possible_edges_for_pair(a, b, yeo_counts)
        master_rows.append({
            "Study": "Grimm 2018",
            "Atlas": "AAL3v1 + target spheres",
            "Yeo_Pair_A": a,
            "Yeo_Pair_B": b,
            "Regions_in_A": nA,
            "Regions_in_B": nB,
            "Possible_Edges": possible,
            "Pair_Type": pair_type,
        })
master_df = pd.DataFrame(master_rows)

# =========================
# Row-level table for reported Grimm edges
# Seed: AAL ROI
# Target: MNI sphere
# Denominator uses AAL atlas Yeo counts
# =========================
df = pd.read_excel(GRIMM_XLSX)
df.columns = df.columns.astype(str).str.replace("\u00a0", " ", regex=False).str.strip()

study_c = "Study"
con_c   = "Contrast"
seed_c  = "Seed"
targ_c  = "Target"
x_c, y_c, z_c = "x (target)", "y (target)", "z (target)"
rad_c   = "Size (target)"
dir_c   = "Direction"

seed_ids = []
seed_labels = []
seed_yeo_list = []
target_radius = []
target_yeo_list = []
target_dice_list = []
pair_a = []
pair_b = []
possible_list = []
pair_type_list = []

for i, r in df.iterrows():
    sid, seed_label_exact, seed_mask = require_aal_mask(aal_img, aal_label_to_id, r[seed_c], row_idx=i)
    seed_yeo, _ = best_yeo_for_mask(seed_mask, yeo_masks)
    if seed_yeo not in YEO7:
        raise ValueError(f"[Seed Yeo mapping error] Row {i}: seed {r[seed_c]!r} mapped to {seed_yeo!r}")

    x, y, z = float(r[x_c]), float(r[y_c]), float(r[z_c])
    rad = float(r[rad_c])
    targ_mask = sphere_mask_on_grid(aal_img, (x, y, z), radius_mm=rad)
    targ_yeo, targ_d = best_yeo_for_mask(targ_mask, yeo_masks)
    if targ_yeo not in YEO7:
        raise ValueError(f"[Target Yeo mapping error] Row {i}: target coord {(x, y, z)} mapped to {targ_yeo!r}")

    A, B = (seed_yeo, targ_yeo) if YEO7.index(seed_yeo) <= YEO7.index(targ_yeo) else (targ_yeo, seed_yeo)
    _, _, possible, pair_type = possible_edges_for_pair(A, B, yeo_counts)

    seed_ids.append(sid)
    seed_labels.append(seed_label_exact)
    seed_yeo_list.append(seed_yeo)
    target_radius.append(rad)
    target_yeo_list.append(targ_yeo)
    target_dice_list.append(targ_d)
    pair_a.append(A)
    pair_b.append(B)
    possible_list.append(possible)
    pair_type_list.append(pair_type)

rowlevel_df = pd.DataFrame({
    "Study": df[study_c].astype(str).values if study_c in df.columns else ["Grimm"] * len(df),
    "Contrast": df[con_c].astype(str).values,
    "Seed_Original": df[seed_c].astype(str).values,
    "Seed_AAL_ID": seed_ids,
    "Seed_AAL_Label": seed_labels,
    "Seed_Yeo7": seed_yeo_list,
    "Target_Original": df[targ_c].astype(str).values,
    "Target_X": df[x_c].values,
    "Target_Y": df[y_c].values,
    "Target_Z": df[z_c].values,
    "Target_Radius_mm": target_radius,
    "Target_Yeo7": target_yeo_list,
    "Target_Dice": target_dice_list,
    "Yeo_Pair_A": pair_a,
    "Yeo_Pair_B": pair_b,
    "Direction": df[dir_c].astype(str).values,
    "Possible_Edges": possible_list,
    "Pair_Type": pair_type_list,
})

# =========================
# Save
# =========================
with pd.ExcelWriter(OUT_XLSX, engine="openpyxl") as writer:
    mapping_df.to_excel(writer, sheet_name="AAL_to_Yeo7_mapping", index=False)
    yeo_counts_df.to_excel(writer, sheet_name="Yeo_counts", index=False)
    master_df.to_excel(writer, sheet_name="Master_possible_edges", index=False)
    rowlevel_df.to_excel(writer, sheet_name="Rowlevel_denominators", index=False)

print("✅ Saved:", OUT_XLSX)
print("Seed atlas used:", AAL_NII)
print("Yeo atlas used:", YEO_NII)
print("\nYeo counts:")
print(yeo_counts_df)
print("\nMaster possible-edge table:")
print(master_df.head(15))

✅ Saved: /Users/maximegodart/Desktop/CBMA_coordinate_extract/CONNECTIVITY/NEW ANALYSIS DATA/possible_edge_outputs/Grimm_2018_possible_edges_master.xlsx
Seed atlas used: /Users/maximegodart/Desktop/CBMA_coordinate_extract/CONNECTIVITY/NEW ANALYSIS DATA/Atlases/AAL3v1_1mm.nii.gz
Yeo atlas used: /Users/maximegodart/Desktop/CBMA_coordinate_extract/CONNECTIVITY/NEW ANALYSIS DATA/Atlases/Yeo2011_7Networks_MNI152_FreeSurferConformed1mm.nii

Yeo counts:
               Yeo7  N_AAL_Regions
0            Visual             87
1       Somatomotor             12
2   DorsalAttention              6
3  VentralAttention             11
4            Limbic             24
5    Frontoparietal              8
6           Default             18

Master possible-edge table:
         Study                    Atlas       Yeo_Pair_A        Yeo_Pair_B  \
0   Grimm 2018  AAL3v1 + target spheres           Visual            Visual   
1   Grimm 2018  AAL3v1 + target spheres           Visual       Somatomotor   
2   Gri

In [6]:
# ============================================
# MADSEN 2021: RAICHLE ROI NETWORKS -> YEO-7 POSSIBLE EDGES
# Creates:
#   1) Raichle-network-to-Yeo mapping from coordinate-defined sphere masks
#   2) Yeo counts based on original Raichle networks
#   3) master table with one row per Yeo pair + denominator
#   4) row-level table for reported edges
# ============================================

import os
import re
import numpy as np
import pandas as pd
import nibabel as nib

# =========================
# Paths
# =========================
BASE_DIR  = "/Users/maximegodart/Desktop/CBMA_coordinate_extract/CONNECTIVITY/NEW ANALYSIS DATA"
ATLAS_DIR = os.path.join(BASE_DIR, "Atlases")

COORD_XLSX = os.path.join(BASE_DIR, "Madsen_2021_Network_Coordinates.xlsx")
EDGES_XLSX = os.path.join(BASE_DIR, "Madsen_2021_significant_edges_FINAL.xlsx")

YEO_NII = os.path.join(ATLAS_DIR, "Yeo2011_7Networks_MNI152_FreeSurferConformed1mm.nii")
if not os.path.exists(YEO_NII):
    raise FileNotFoundError(f"Could not find Yeo atlas: {YEO_NII}")

OUT_DIR  = os.path.join(BASE_DIR, "possible_edge_outputs")
os.makedirs(OUT_DIR, exist_ok=True)
OUT_XLSX = os.path.join(OUT_DIR, "Madsen_2021_possible_edges_master.xlsx")

RADIUS_MM = 10.0

YEO7 = [
    "Visual",
    "Somatomotor",
    "DorsalAttention",
    "VentralAttention",
    "Limbic",
    "Frontoparietal",
    "Default",
]

# =========================
# Helpers
# =========================
def clean_ws(s):
    return re.sub(r"\s+", " ", str(s).strip())

def norm_key(s):
    return clean_ws(s).lower().replace(" ", "")

def dice(a, b):
    inter = np.count_nonzero(a & b)
    na = np.count_nonzero(a)
    nb = np.count_nonzero(b)
    return 0.0 if (na + nb) == 0 else (2.0 * inter) / (na + nb)

def sphere_mask(ref_img, coord, radius_mm):
    shape = ref_img.shape[:3]
    affine = ref_img.affine
    inv = np.linalg.inv(affine)

    cx, cy, cz = nib.affines.apply_affine(inv, np.array(coord, float))
    cx, cy, cz = float(cx), float(cy), float(cz)

    vx = float(np.linalg.norm(affine[:3, 0]))
    vy = float(np.linalg.norm(affine[:3, 1]))
    vz = float(np.linalg.norm(affine[:3, 2]))

    rx = int(np.ceil(radius_mm / vx))
    ry = int(np.ceil(radius_mm / vy))
    rz = int(np.ceil(radius_mm / vz))

    x0 = max(0, int(np.floor(cx)) - rx)
    x1 = min(shape[0], int(np.floor(cx)) + rx + 1)
    y0 = max(0, int(np.floor(cy)) - ry)
    y1 = min(shape[1], int(np.floor(cy)) + ry + 1)
    z0 = max(0, int(np.floor(cz)) - rz)
    z1 = min(shape[2], int(np.floor(cz)) + rz + 1)

    xs = np.arange(x0, x1)
    ys = np.arange(y0, y1)
    zs = np.arange(z0, z1)
    X, Y, Z = np.meshgrid(xs, ys, zs, indexing="ij")

    dx = (X - cx) * vx
    dy = (Y - cy) * vy
    dz = (Z - cz) * vz
    local = (dx * dx + dy * dy + dz * dz) <= (radius_mm ** 2)

    mask = np.zeros(shape, dtype=bool)
    mask[x0:x1, y0:y1, z0:z1] = local
    return mask

def best_yeo_for_mask(mask, yeo_masks):
    best_name, best_d, best_inter = None, -1.0, -1
    for nm, ymask in yeo_masks.items():
        inter = np.count_nonzero(mask & ymask)
        d = dice(mask, ymask)
        if (d > best_d) or (np.isclose(d, best_d) and inter > best_inter):
            best_name, best_d, best_inter = nm, float(d), int(inter)
    return best_name, best_d

def possible_edges_for_pair(a, b, yeo_counts):
    nA = int(yeo_counts.get(a, 0))
    nB = int(yeo_counts.get(b, 0))
    if a == b:
        return nA, nB, nA * (nA - 1) // 2, "within"
    return nA, nB, nA * nB, "between"

# =========================
# Load coordinate table
# =========================
coords = pd.read_excel(COORD_XLSX)
coords.columns = coords.columns.astype(str).str.replace("\u00a0", " ", regex=False).str.strip()

required_coord_cols = {"Network", "X", "Y", "Z"}
missing = required_coord_cols - set(coords.columns)
if missing:
    raise ValueError(f"Coordinate file missing required columns: {sorted(missing)}")

coords["Network_clean"] = coords["Network"].map(clean_ws)
coords["Network_norm"] = coords["Network_clean"].map(norm_key)

# =========================
# Load Yeo atlas and masks
# =========================
yeo_img = nib.load(YEO_NII)
yeo = yeo_img.get_fdata().astype(int)

if yeo.ndim == 4 and yeo.shape[-1] == 1:
    yeo = np.squeeze(yeo, axis=-1)

yeo_ids = sorted([i for i in np.unique(yeo) if i != 0])[:7]
yeo_masks = {}
for i, yid in enumerate(yeo_ids):
    yeo_masks[YEO7[i]] = (yeo == yid)

# =========================
# Build Raichle network masks from union of 10 mm spheres
# =========================
raichle_masks = {}
for network in coords["Network_clean"].unique():
    sub = coords[coords["Network_clean"] == network]
    mask = np.zeros(yeo_img.shape[:3], dtype=bool)
    for _, r in sub.iterrows():
        mask |= sphere_mask(yeo_img, (r["X"], r["Y"], r["Z"]), RADIUS_MM)
    raichle_masks[network] = mask

# =========================
# Map original Raichle networks -> Yeo-7
# =========================
raichle_to_yeo = {}
raichle_to_dice = {}
for net, mask in raichle_masks.items():
    best, best_d = best_yeo_for_mask(mask, yeo_masks)
    raichle_to_yeo[net] = best
    raichle_to_dice[net] = best_d

mapping_df = pd.DataFrame({
    "Raichle_Network": list(raichle_to_yeo.keys()),
    "Raichle_Network_norm": [norm_key(x) for x in raichle_to_yeo.keys()],
    "Yeo7": [raichle_to_yeo[x] for x in raichle_to_yeo.keys()],
    "Dice": [raichle_to_dice[x] for x in raichle_to_yeo.keys()],
    "N_Coordinates": [int((coords["Network_clean"] == x).sum()) for x in raichle_to_yeo.keys()],
}).sort_values("Raichle_Network").reset_index(drop=True)

# =========================
# Yeo counts from original Raichle networks
# =========================
yeo_counts = (
    mapping_df.groupby("Yeo7", as_index=False)["Raichle_Network"]
    .nunique()
    .rename(columns={"Raichle_Network": "N_Raichle_Networks"})
)
yeo_counts = yeo_counts.set_index("Yeo7")["N_Raichle_Networks"].to_dict()
yeo_counts = {k: int(yeo_counts.get(k, 0)) for k in YEO7}

yeo_counts_df = pd.DataFrame({
    "Yeo7": YEO7,
    "N_Raichle_Networks": [yeo_counts[k] for k in YEO7]
})

# =========================
# Master table: one row per Yeo pair
# =========================
master_rows = []
for i, a in enumerate(YEO7):
    for b in YEO7[i:]:
        nA, nB, possible, pair_type = possible_edges_for_pair(a, b, yeo_counts)
        master_rows.append({
            "Study": "Madsen 2021",
            "Atlas": "Raichle ROI networks",
            "Yeo_Pair_A": a,
            "Yeo_Pair_B": b,
            "Regions_in_A": nA,
            "Regions_in_B": nB,
            "Possible_Edges": possible,
            "Pair_Type": pair_type,
        })
master_df = pd.DataFrame(master_rows)

# =========================
# Edge-table abbreviations in Madsen file
# =========================
ABBREV_MAP = {
    "DMN": "Default mode network",
    "DAN": "Dorsal attention network",
    "ECN": "Executive control network",
    "SAN": "Salience network",
    "SMN": "Sensorimotor network",
    "VN": "Visual network",
    "AN": "Auditory network",
}

# Robust normalized aliases in case coord file uses different spacing/casing
NETWORK_ALIASES = {
    "defaultmodenetwork": "Default mode network",
    "dorsalattentionnetwork": "Dorsal attention network",
    "executivecontrolnetwork": "Executive control network",
    "saliencenetwork": "Salience network",
    "sensorimotornetwork": "Sensorimotor network",
    "visualnetwork": "Visual network",
    "auditorynetwork": "Auditory network",
}

# add coord-file networks themselves as valid aliases
for n in coords["Network_clean"].unique():
    NETWORK_ALIASES[norm_key(n)] = n

# =========================
# Convert row-level edges and attach denominators
# =========================
edges = pd.read_excel(EDGES_XLSX)
edges.columns = edges.columns.astype(str).str.replace("\u00a0", " ", regex=False).str.strip()

required_edge_cols = {"Seed", "Target", "Contrast", "Direction"}
missing = required_edge_cols - set(edges.columns)
if missing:
    raise ValueError(f"Edge file missing required columns: {sorted(missing)}")

def resolve_raichle_label(x, row_idx, colname):
    raw = clean_ws(x)
    # first expand abbreviation if present
    raw = ABBREV_MAP.get(raw, raw)
    key = norm_key(raw)
    if key not in NETWORK_ALIASES:
        raise ValueError(
            f"[Madsen mapping error] Row {row_idx}: could not resolve {colname} label {x!r} -> {raw!r}"
        )
    canonical = NETWORK_ALIASES[key]
    if canonical not in raichle_to_yeo:
        raise ValueError(
            f"[Madsen mapping error] Row {row_idx}: resolved {colname} label {x!r} -> {canonical!r}, "
            f"but that network was not found in the coordinate file."
        )
    return canonical

seed_raichle = []
targ_raichle = []
seed_yeo = []
targ_yeo = []
pair_a = []
pair_b = []
possible_edges = []
pair_type_list = []

for i, r in edges.iterrows():
    s_net = resolve_raichle_label(r["Seed"], i, "Seed")
    t_net = resolve_raichle_label(r["Target"], i, "Target")
    s_yeo = raichle_to_yeo[s_net]
    t_yeo = raichle_to_yeo[t_net]

    A, B = (s_yeo, t_yeo) if YEO7.index(s_yeo) <= YEO7.index(t_yeo) else (t_yeo, s_yeo)
    _, _, denom, ptype = possible_edges_for_pair(A, B, yeo_counts)

    seed_raichle.append(s_net)
    targ_raichle.append(t_net)
    seed_yeo.append(s_yeo)
    targ_yeo.append(t_yeo)
    pair_a.append(A)
    pair_b.append(B)
    possible_edges.append(denom)
    pair_type_list.append(ptype)

rowlevel_df = pd.DataFrame({
    "Study": "Madsen",
    "Contrast": edges["Contrast"].astype(str).values,
    "Seed_Original": edges["Seed"].astype(str).values,
    "Target_Original": edges["Target"].astype(str).values,
    "Seed_Raichle_Network": seed_raichle,
    "Target_Raichle_Network": targ_raichle,
    "Seed_Yeo7": seed_yeo,
    "Target_Yeo7": targ_yeo,
    "Yeo_Pair_A": pair_a,
    "Yeo_Pair_B": pair_b,
    "Direction": edges["Direction"].astype(str).values,
    "Possible_Edges": possible_edges,
    "Pair_Type": pair_type_list,
})

# =========================
# Save
# =========================
with pd.ExcelWriter(OUT_XLSX, engine="openpyxl") as writer:
    mapping_df.to_excel(writer, sheet_name="Raichle_to_Yeo7_mapping", index=False)
    yeo_counts_df.to_excel(writer, sheet_name="Yeo_counts", index=False)
    master_df.to_excel(writer, sheet_name="Master_possible_edges", index=False)
    rowlevel_df.to_excel(writer, sheet_name="Rowlevel_denominators", index=False)

print("✅ Saved:", OUT_XLSX)
print("\nRaichle -> Yeo mapping:")
print(mapping_df)
print("\nYeo counts:")
print(yeo_counts_df)
print("\nMaster possible-edge table:")
print(master_df.head(15))

✅ Saved: /Users/maximegodart/Desktop/CBMA_coordinate_extract/CONNECTIVITY/NEW ANALYSIS DATA/possible_edge_outputs/Madsen_2021_possible_edges_master.xlsx

Raichle -> Yeo mapping:
             Raichle_Network     Raichle_Network_norm              Yeo7  \
0           Auditory network          auditorynetwork       Somatomotor   
1       Default mode network       defaultmodenetwork           Default   
2   Dorsal attention network   dorsalattentionnetwork   DorsalAttention   
3  Executive control network  executivecontrolnetwork    Frontoparietal   
4           Salience network          saliencenetwork  VentralAttention   
5       Sensorimotor network      sensorimotornetwork       Somatomotor   
6             Visual network            visualnetwork            Visual   

       Dice  N_Coordinates  
0  0.064233              2  
1  0.136074              9  
2  0.230034              8  
3  0.150192              5  
4  0.203933              7  
5  0.068895              3  
6  0.000000       

In [7]:
# ============================================
# MASON 2020: SMITH RSN10 -> YEO-7 POSSIBLE EDGES
# Creates:
#   1) Smith RSN10 component-to-Yeo mapping from thresholded atlas overlap
#   2) Yeo counts based on original Smith RSN10 components
#   3) master table with one row per Yeo pair + denominator
#   4) row-level table for reported Mason edges
# ============================================

import os
import re
import numpy as np
import pandas as pd
import nibabel as nib
from nilearn.image import resample_to_img

# =========================
# Paths
# =========================
BASE_DIR  = "/Users/maximegodart/Desktop/CBMA_coordinate_extract/CONNECTIVITY/NEW ANALYSIS DATA"
ATLAS_DIR = os.path.join(BASE_DIR, "Atlases")

MASON_XLSX   = os.path.join(BASE_DIR, "Mason_2020_significant_edges_FINAL.xlsx")
SMITH_NII    = os.path.join(ATLAS_DIR, "PNAS_Smith09_rsn10.nii.gz")
SMITH_LABELS = os.path.join(ATLAS_DIR, "SMITH09_RSN10_labels.txt")
YEO_NII      = os.path.join(ATLAS_DIR, "Yeo2011_7Networks_MNI152_FreeSurferConformed1mm.nii")

if not os.path.exists(SMITH_NII):
    raise FileNotFoundError(f"Could not find Smith atlas: {SMITH_NII}")
if not os.path.exists(SMITH_LABELS):
    raise FileNotFoundError(f"Could not find Smith labels: {SMITH_LABELS}")
if not os.path.exists(YEO_NII):
    raise FileNotFoundError(f"Could not find Yeo atlas: {YEO_NII}")

OUT_DIR = os.path.join(BASE_DIR, "possible_edge_outputs")
os.makedirs(OUT_DIR, exist_ok=True)
OUT_XLSX = os.path.join(OUT_DIR, "Mason_2020_possible_edges_master.xlsx")

YEO7 = ["Visual","Somatomotor","DorsalAttention","VentralAttention","Limbic","Frontoparietal","Default"]

# IMPORTANT: threshold Smith ICA maps before masks
SMITH_Z_THRESHOLD = 2.3

# =========================
# Helpers
# =========================
def clean_ws(s):
    return re.sub(r"\s+", " ", str(s).strip())

def norm(s):
    s = clean_ws(s).lower()
    s = re.sub(r"[^a-z0-9]+", "", s)
    return s

def dice(a, b):
    inter = np.count_nonzero(a & b)
    na = np.count_nonzero(a)
    nb = np.count_nonzero(b)
    return 0.0 if (na + nb) == 0 else (2.0 * inter) / (na + nb)

def possible_edges_for_pair(a, b, yeo_counts):
    nA = int(yeo_counts.get(a, 0))
    nB = int(yeo_counts.get(b, 0))
    if a == b:
        return nA, nB, nA * (nA - 1) // 2, "within"
    return nA, nB, nA * nB, "between"

# =========================
# Load Smith RSN10 labels
# =========================
def load_smith_labels(labels_path):
    id_to_label = {}
    with open(labels_path, "r", encoding="utf-8", errors="ignore") as f:
        for ln in f:
            ln = ln.strip()
            if not ln:
                continue
            m = re.match(r"^(\d+)\s+(.+)$", ln)
            if not m:
                continue
            rid = int(m.group(1))
            lab = clean_ws(m.group(2))
            id_to_label[rid] = lab
    label_to_id = {norm(v): k for k, v in id_to_label.items()}
    return id_to_label, label_to_id

SMITH_ALIAS = {
    "cerebellum": "Cerebellum",
    "auditory": "Auditory",
    "aud": "Auditory",
    "sensorimotor": "Sensorimotor",
    "sm": "Sensorimotor",
    "dmn": "Default mode network",
    "defaultmodenetwork": "Default mode network",
    "executive": "Executive control",
    "executivecontrol": "Executive control",
    "ecn": "Executive control",
    "lfp": "Left frontoparietal",
    "leftfrontoparietal": "Left frontoparietal",
    "frontoparietal1": "Left frontoparietal",
    "rfp": "Right frontoparietal",
    "rightfrontoparietal": "Right frontoparietal",
    "frontoparietal2": "Right frontoparietal",
    "visual1": "Visual medial",
    "visual2": "Visual occipital pole",
    "visual3": "Visual lateral",
    "vism": "Visual medial",
    "viso": "Visual occipital pole",
    "visl": "Visual lateral",
    "dan": "Executive control",
    "dan2": "Executive control",
    "dmn2": "Default mode network",
}

def canonical_label(x, row_idx, colname, study_name, id_to_label):
    raw = clean_ws(x)
    k = norm(raw)

    for rid, lab in id_to_label.items():
        if norm(lab) == k:
            return lab

    if k in SMITH_ALIAS:
        canon = SMITH_ALIAS[k]
        for rid, lab in id_to_label.items():
            if norm(lab) == norm(canon):
                return lab

    hints = [lab for rid, lab in id_to_label.items() if k in norm(lab) or norm(lab) in k][:10]
    hint_txt = "\nClosest matches:\n  - " + "\n  - ".join(hints) if hints else ""
    raise ValueError(
        f"[{study_name}] Row {row_idx}: could not map '{colname}' label to Smith RSN10.\n"
        f"  Raw value: {raw!r}\n"
        f"  Normalized: {k!r}{hint_txt}"
    )

# =========================
# Smith RSN10 -> Yeo-7 via thresholded Dice
# =========================
def compute_smith_to_yeo7(smith_nii_path, yeo_nii_path, zthr=2.3):
    smith_img = nib.load(smith_nii_path)
    yeo_img   = nib.load(yeo_nii_path)

    smith = smith_img.get_fdata()
    if smith.ndim != 4:
        raise ValueError(f"Expected Smith atlas to be 4D. Got shape: {smith.shape}")
    if smith.shape[-1] < 10:
        raise ValueError(f"Expected >=10 Smith components. Got: {smith.shape[-1]}")

    yd = yeo_img.get_fdata()
    if yd.ndim == 4 and yd.shape[-1] == 1:
        yeo_img = nib.Nifti1Image(np.squeeze(yd, axis=-1), affine=yeo_img.affine)

    yeo_rs = resample_to_img(yeo_img, smith_img, interpolation="nearest")
    yeo = yeo_rs.get_fdata().astype(int)

    yeo_ids = sorted([i for i in np.unique(yeo) if i != 0])[:7]
    yeo_id_to_name = {yeo_ids[i]: YEO7[i] for i in range(min(7, len(yeo_ids)))}

    smith_to_yeo = {}
    smith_to_dice = {}
    mapping_rows = []

    for k in range(10):
        comp = smith[..., k]
        comp_mask = comp > zthr

        if np.count_nonzero(comp_mask) == 0:
            raise ValueError(
                f"Smith component {k+1} has zero voxels above threshold {zthr}. "
                f"Try lowering SMITH_Z_THRESHOLD."
            )

        best_name, best_d, best_inter = None, -1.0, -1
        for yid in yeo_ids:
            ym = (yeo == yid)
            inter = np.count_nonzero(comp_mask & ym)
            d = dice(comp_mask, ym)
            if (d > best_d) or (np.isclose(d, best_d) and inter > best_inter):
                best_d, best_inter = float(d), int(inter)
                best_name = yeo_id_to_name.get(yid)

        comp_id = k + 1
        smith_to_yeo[comp_id] = best_name
        smith_to_dice[comp_id] = best_d
        mapping_rows.append({
            "Smith_Component": comp_id,
            "Best_Yeo7": best_name,
            "Dice": best_d,
            "Intersect_Voxels": best_inter,
            "Component_Voxels": int(np.count_nonzero(comp_mask)),
        })

    mapping_df = pd.DataFrame(mapping_rows)
    return smith_to_yeo, smith_to_dice, mapping_df

# =========================
# Build mapping + Yeo counts
# =========================
id_to_label, label_to_id = load_smith_labels(SMITH_LABELS)
smith_to_yeo, smith_to_dice, overlap_df = compute_smith_to_yeo7(SMITH_NII, YEO_NII, zthr=SMITH_Z_THRESHOLD)

mapping_rows = []
for comp_id in sorted(id_to_label.keys()):
    mapping_rows.append({
        "Smith_Component": comp_id,
        "Smith_Label": id_to_label[comp_id],
        "Yeo7": smith_to_yeo.get(comp_id),
        "Dice": smith_to_dice.get(comp_id),
    })
mapping_df = pd.DataFrame(mapping_rows)

yeo_counts = (
    mapping_df.groupby("Yeo7", as_index=False)["Smith_Component"]
    .nunique()
    .rename(columns={"Smith_Component": "N_Smith_Components"})
)
yeo_counts = yeo_counts.set_index("Yeo7")["N_Smith_Components"].to_dict()
yeo_counts = {k: int(yeo_counts.get(k, 0)) for k in YEO7}

yeo_counts_df = pd.DataFrame({
    "Yeo7": YEO7,
    "N_Smith_Components": [yeo_counts[k] for k in YEO7]
})

# =========================
# Master table: one row per Yeo pair
# =========================
master_rows = []
for i, a in enumerate(YEO7):
    for b in YEO7[i:]:
        nA, nB, possible, pair_type = possible_edges_for_pair(a, b, yeo_counts)
        master_rows.append({
            "Study": "Mason 2020",
            "Atlas": "Smith RSN10",
            "Yeo_Pair_A": a,
            "Yeo_Pair_B": b,
            "Regions_in_A": nA,
            "Regions_in_B": nB,
            "Possible_Edges": possible,
            "Pair_Type": pair_type,
        })
master_df = pd.DataFrame(master_rows)

# =========================
# Row-level table for Mason edges
# =========================
df = pd.read_excel(MASON_XLSX)
df.columns = df.columns.astype(str).str.replace("\u00a0", " ", regex=False).str.strip()

seed_c, targ_c, dir_c = "Seed", "Target", "Direction"
con_c = "Contrast"
net_c = "Network"

if net_c in df.columns:
    net_norm = (
        df[net_c].astype(str).str.strip().str.lower()
        .str.replace(r"[^a-z0-9]+", "", regex=True)
    )
    sub = df[net_norm.isin(["smith2009", "smith", "smithrsn10"])].copy()
    if len(sub) == 0:
        sub = df.copy()
else:
    sub = df.copy()

seed_label_exact = []
targ_label_exact = []
seed_comp = []
targ_comp = []
seed_yeo = []
targ_yeo = []
pair_a = []
pair_b = []
possible_edges = []
pair_type_list = []

for i, r in sub.iterrows():
    s_lab = canonical_label(r[seed_c], i, "Seed", "Mason", id_to_label)
    t_lab = canonical_label(r[targ_c], i, "Target", "Mason", id_to_label)

    s_id = label_to_id[norm(s_lab)]
    t_id = label_to_id[norm(t_lab)]
    s_yeo = smith_to_yeo.get(s_id)
    t_yeo = smith_to_yeo.get(t_id)

    if s_yeo not in YEO7:
        raise ValueError(f"[Mason] Row {i}: Seed {s_lab!r} mapped to invalid Yeo value: {s_yeo!r}")
    if t_yeo not in YEO7:
        raise ValueError(f"[Mason] Row {i}: Target {t_lab!r} mapped to invalid Yeo value: {t_yeo!r}")

    A, B = (s_yeo, t_yeo) if YEO7.index(s_yeo) <= YEO7.index(t_yeo) else (t_yeo, s_yeo)
    _, _, denom, ptype = possible_edges_for_pair(A, B, yeo_counts)

    seed_label_exact.append(s_lab)
    targ_label_exact.append(t_lab)
    seed_comp.append(s_id)
    targ_comp.append(t_id)
    seed_yeo.append(s_yeo)
    targ_yeo.append(t_yeo)
    pair_a.append(A)
    pair_b.append(B)
    possible_edges.append(denom)
    pair_type_list.append(ptype)

rowlevel_df = pd.DataFrame({
    "Study": "Mason",
    "Contrast": sub[con_c].astype(str).values,
    "Seed_Original": sub[seed_c].astype(str).values,
    "Target_Original": sub[targ_c].astype(str).values,
    "Seed_Smith_Label": seed_label_exact,
    "Target_Smith_Label": targ_label_exact,
    "Seed_Smith_Component": seed_comp,
    "Target_Smith_Component": targ_comp,
    "Seed_Yeo7": seed_yeo,
    "Target_Yeo7": targ_yeo,
    "Yeo_Pair_A": pair_a,
    "Yeo_Pair_B": pair_b,
    "Direction": sub[dir_c].astype(str).values,
    "Possible_Edges": possible_edges,
    "Pair_Type": pair_type_list,
})

# =========================
# Save
# =========================
with pd.ExcelWriter(OUT_XLSX, engine="openpyxl") as writer:
    mapping_df.to_excel(writer, sheet_name="Smith_to_Yeo7_mapping", index=False)
    overlap_df.to_excel(writer, sheet_name="Smith_overlap_debug", index=False)
    yeo_counts_df.to_excel(writer, sheet_name="Yeo_counts", index=False)
    master_df.to_excel(writer, sheet_name="Master_possible_edges", index=False)
    rowlevel_df.to_excel(writer, sheet_name="Rowlevel_denominators", index=False)

print("✅ Saved:", OUT_XLSX)
print("\nSmith -> Yeo mapping:")
print(mapping_df)
print("\nYeo counts:")
print(yeo_counts_df)
print("\nMaster possible-edge table:")
print(master_df.head(15))

✅ Saved: /Users/maximegodart/Desktop/CBMA_coordinate_extract/CONNECTIVITY/NEW ANALYSIS DATA/possible_edge_outputs/Mason_2020_possible_edges_master.xlsx

Smith -> Yeo mapping:
   Smith_Component            Smith_Label              Yeo7      Dice
0                1          Visual medial            Visual  0.278647
1                2  Visual occipital pole            Visual  0.198226
2                3         Visual lateral            Visual  0.234508
3                4   Default mode network           Default  0.342841
4                5             Cerebellum            Visual  0.062025
5                6           Sensorimotor       Somatomotor  0.244239
6                7               Auditory  VentralAttention  0.217414
7                8      Executive control    Frontoparietal  0.146063
8                9    Left frontoparietal    Frontoparietal  0.219659
9               10   Right frontoparietal    Frontoparietal  0.183292

Yeo counts:
               Yeo7  N_Smith_Components
0 

In [9]:
# ============================================
# PALHANO-FONTES 2015: MNI SEED SPHERES + MNI TARGET SPHERES -> YEO-7 POSSIBLE EDGES
# Creates:
#   1) seed-sphere and target-sphere Yeo-7 assignments for reported edges
#   2) Yeo counts based on original sphere-defined nodes in the study
#   3) master table with one row per Yeo pair + denominator
#   4) row-level table for reported edges
#
# NOTE:
# For this study, the "original atlas units" are the reported sphere-defined nodes
# (seed spheres and target spheres), since there is no external parcel atlas.
# Denominators are therefore based on the count of unique original study nodes
# assigned to each Yeo-7 network.
# ============================================

import os
import numpy as np
import pandas as pd
import nibabel as nib

# =========================
# Paths
# =========================
BASE_DIR  = "/Users/maximegodart/Desktop/CBMA_coordinate_extract/CONNECTIVITY/NEW ANALYSIS DATA"
ATLAS_DIR = os.path.join(BASE_DIR, "Atlases")

PALHANO_XLSX = os.path.join(BASE_DIR, "PalhanoFontes_2015_significant edges_FINAL.xlsx")
YEO_NII      = os.path.join(ATLAS_DIR, "Yeo2011_7Networks_MNI152_FreeSurferConformed1mm.nii")

if not os.path.exists(YEO_NII):
    raise FileNotFoundError(f"Could not find Yeo atlas: {YEO_NII}")

OUT_DIR  = os.path.join(BASE_DIR, "possible_edge_outputs")
os.makedirs(OUT_DIR, exist_ok=True)
OUT_XLSX = os.path.join(OUT_DIR, "PalhanoFontes_2015_possible_edges_master.xlsx")

YEO7 = ["Visual","Somatomotor","DorsalAttention","VentralAttention","Limbic","Frontoparietal","Default"]

# =========================
# Helpers
# =========================
def clean_ws(s):
    return " ".join(str(s).strip().split())

def dice(a, b):
    inter = np.count_nonzero(a & b)
    na = np.count_nonzero(a)
    nb = np.count_nonzero(b)
    return 0.0 if (na + nb) == 0 else (2.0 * inter) / (na + nb)

def sphere_mask_on_grid(ref_img: nib.Nifti1Image, center_xyz_mm, radius_mm=10.0):
    """Create a boolean sphere mask in the voxel grid of ref_img."""
    shape = ref_img.shape[:3]
    aff = ref_img.affine
    inv = np.linalg.inv(aff)

    cx, cy, cz = nib.affines.apply_affine(inv, np.array(center_xyz_mm, dtype=float))
    cx, cy, cz = float(cx), float(cy), float(cz)

    vx = float(np.linalg.norm(aff[:3, 0]))
    vy = float(np.linalg.norm(aff[:3, 1]))
    vz = float(np.linalg.norm(aff[:3, 2]))

    rx = int(np.ceil(radius_mm / vx))
    ry = int(np.ceil(radius_mm / vy))
    rz = int(np.ceil(radius_mm / vz))

    x0 = max(0, int(np.floor(cx)) - rx)
    x1 = min(shape[0], int(np.floor(cx)) + rx + 1)
    y0 = max(0, int(np.floor(cy)) - ry)
    y1 = min(shape[1], int(np.floor(cy)) + ry + 1)
    z0 = max(0, int(np.floor(cz)) - rz)
    z1 = min(shape[2], int(np.floor(cz)) + rz + 1)

    xs = np.arange(x0, x1)
    ys = np.arange(y0, y1)
    zs = np.arange(z0, z1)
    X, Y, Z = np.meshgrid(xs, ys, zs, indexing="ij")

    dx = (X - cx) * vx
    dy = (Y - cy) * vy
    dz = (Z - cz) * vz

    local = (dx * dx + dy * dy + dz * dz) <= (radius_mm ** 2)

    mask = np.zeros(shape, dtype=bool)
    mask[x0:x1, y0:y1, z0:z1] = local
    return mask

def get_yeo_masks(yeo_img):
    yd = yeo_img.get_fdata()
    if yd.ndim == 4 and yd.shape[-1] == 1:
        yd = np.squeeze(yd, axis=-1)
        yeo_img = nib.Nifti1Image(yd, affine=yeo_img.affine)

    yeo = yd.astype(int)
    yeo_ids = sorted([i for i in np.unique(yeo) if i != 0])[:7]

    yeo_masks = {}
    for i, yid in enumerate(yeo_ids):
        yeo_masks[YEO7[i]] = (yeo == yid)
    return yeo_masks

def best_yeo_for_mask(mask, yeo_masks):
    best_name = None
    best_d = -1.0
    best_inter = -1

    for name, ym in yeo_masks.items():
        inter = np.count_nonzero(mask & ym)
        d = dice(mask, ym)
        if (d > best_d) or (np.isclose(d, best_d) and inter > best_inter):
            best_name = name
            best_d = float(d)
            best_inter = int(inter)

    return best_name, best_d, best_inter

def possible_edges_for_pair(a, b, yeo_counts):
    nA = int(yeo_counts.get(a, 0))
    nB = int(yeo_counts.get(b, 0))
    if a == b:
        return nA, nB, nA * (nA - 1) // 2, "within"
    return nA, nB, nA * nB, "between"

# =========================
# Load data + atlas
# =========================
df = pd.read_excel(PALHANO_XLSX)
df.columns = df.columns.astype(str).str.replace("\u00a0", " ", regex=False).str.strip()

yeo_img = nib.load(YEO_NII)
yeo_masks = get_yeo_masks(yeo_img)

# Expected columns
seed_x_c = "x (seed)"
seed_y_c = "y (seed)"
seed_z_c = "z (seed)"
seed_r_c = "Size (seed)"

targ_x_c = "x (target)"
targ_y_c = "y (target)"
targ_z_c = "z (target)"
targ_r_c = "Size (target)"

study_c = "Study"
con_c   = "Contrast"
dir_c   = "Direction"

required_cols = {
    seed_x_c, seed_y_c, seed_z_c, seed_r_c,
    targ_x_c, targ_y_c, targ_z_c, targ_r_c,
    con_c, dir_c
}
missing = required_cols - set(df.columns)
if missing:
    raise ValueError(f"Input file missing required columns: {sorted(missing)}")

# =========================
# Build original node table
# Treat unique seed spheres and unique target spheres as original study nodes
# =========================
seed_nodes = (
    df[[seed_x_c, seed_y_c, seed_z_c, seed_r_c]]
    .drop_duplicates()
    .copy()
    .reset_index(drop=True)
)
seed_nodes["Node_Type"] = "Seed"
seed_nodes["Original_Node_ID"] = [f"S{i+1}" for i in range(len(seed_nodes))]
seed_nodes["Coord_Key"] = (
    seed_nodes[seed_x_c].astype(float).round(6).astype(str) + "|" +
    seed_nodes[seed_y_c].astype(float).round(6).astype(str) + "|" +
    seed_nodes[seed_z_c].astype(float).round(6).astype(str) + "|" +
    seed_nodes[seed_r_c].astype(float).round(6).astype(str)
)

target_nodes = (
    df[[targ_x_c, targ_y_c, targ_z_c, targ_r_c]]
    .drop_duplicates()
    .copy()
    .reset_index(drop=True)
)
target_nodes["Node_Type"] = "Target"
target_nodes["Original_Node_ID"] = [f"T{i+1}" for i in range(len(target_nodes))]
target_nodes["Coord_Key"] = (
    target_nodes[targ_x_c].astype(float).round(6).astype(str) + "|" +
    target_nodes[targ_y_c].astype(float).round(6).astype(str) + "|" +
    target_nodes[targ_z_c].astype(float).round(6).astype(str) + "|" +
    target_nodes[targ_r_c].astype(float).round(6).astype(str)
)

# Harmonize column names for one combined node table
seed_nodes_std = seed_nodes.rename(columns={
    seed_x_c: "X",
    seed_y_c: "Y",
    seed_z_c: "Z",
    seed_r_c: "Radius_mm",
})
target_nodes_std = target_nodes.rename(columns={
    targ_x_c: "X",
    targ_y_c: "Y",
    targ_z_c: "Z",
    targ_r_c: "Radius_mm",
})

node_df = pd.concat([
    seed_nodes_std[["Original_Node_ID", "Node_Type", "X", "Y", "Z", "Radius_mm", "Coord_Key"]],
    target_nodes_std[["Original_Node_ID", "Node_Type", "X", "Y", "Z", "Radius_mm", "Coord_Key"]],
], axis=0, ignore_index=True)

# =========================
# Map each original node sphere -> Yeo-7
# =========================
node_yeo = []
node_dice = []
node_inter = []

for i, r in node_df.iterrows():
    xyz = (float(r["X"]), float(r["Y"]), float(r["Z"]))
    rad = float(r["Radius_mm"])
    mask = sphere_mask_on_grid(yeo_img, xyz, radius_mm=rad)
    yeo_name, d, inter = best_yeo_for_mask(mask, yeo_masks)

    if yeo_name not in YEO7:
        raise ValueError(
            f"[Node mapping error] Row {i}: node {r['Original_Node_ID']} "
            f"coord {xyz} radius {rad} mapped to invalid Yeo label {yeo_name!r}"
        )

    node_yeo.append(yeo_name)
    node_dice.append(d)
    node_inter.append(inter)

node_df["Yeo7"] = node_yeo
node_df["Dice"] = node_dice
node_df["Intersect_Voxels"] = node_inter

# =========================
# Yeo counts from original study nodes
# =========================
yeo_counts = (
    node_df.groupby("Yeo7", as_index=False)["Original_Node_ID"]
    .nunique()
    .rename(columns={"Original_Node_ID": "N_Original_Nodes"})
)
yeo_counts = yeo_counts.set_index("Yeo7")["N_Original_Nodes"].to_dict()
yeo_counts = {k: int(yeo_counts.get(k, 0)) for k in YEO7}

yeo_counts_df = pd.DataFrame({
    "Yeo7": YEO7,
    "N_Original_Nodes": [yeo_counts[k] for k in YEO7]
})

# =========================
# Master table: one row per Yeo pair
# =========================
master_rows = []
for i, a in enumerate(YEO7):
    for b in YEO7[i:]:
        nA, nB, possible, pair_type = possible_edges_for_pair(a, b, yeo_counts)
        master_rows.append({
            "Study": "Palhano-Fontes 2015",
            "Atlas": "Study-defined MNI spheres",
            "Yeo_Pair_A": a,
            "Yeo_Pair_B": b,
            "Regions_in_A": nA,
            "Regions_in_B": nB,
            "Possible_Edges": possible,
            "Pair_Type": pair_type,
        })
master_df = pd.DataFrame(master_rows)

# =========================
# Row-level table for reported edges
# =========================
seed_lookup = node_df[node_df["Node_Type"] == "Seed"].copy()
target_lookup = node_df[node_df["Node_Type"] == "Target"].copy()

seed_lookup = seed_lookup.set_index("Coord_Key")
target_lookup = target_lookup.set_index("Coord_Key")

seed_ids = []
target_ids = []
seed_yeo_out = []
target_yeo_out = []
pair_a = []
pair_b = []
possible_edges = []
pair_type_list = []

for i, r in df.iterrows():
    skey = (
        str(round(float(r[seed_x_c]), 6)) + "|" +
        str(round(float(r[seed_y_c]), 6)) + "|" +
        str(round(float(r[seed_z_c]), 6)) + "|" +
        str(round(float(r[seed_r_c]), 6))
    )
    tkey = (
        str(round(float(r[targ_x_c]), 6)) + "|" +
        str(round(float(r[targ_y_c]), 6)) + "|" +
        str(round(float(r[targ_z_c]), 6)) + "|" +
        str(round(float(r[targ_r_c]), 6))
    )

    if skey not in seed_lookup.index:
        raise ValueError(f"[Seed lookup error] Row {i}: could not find seed node for key {skey}")
    if tkey not in target_lookup.index:
        raise ValueError(f"[Target lookup error] Row {i}: could not find target node for key {tkey}")

    srow = seed_lookup.loc[skey]
    trow = target_lookup.loc[tkey]

    s_id = srow["Original_Node_ID"]
    t_id = trow["Original_Node_ID"]
    s_yeo = srow["Yeo7"]
    t_yeo = trow["Yeo7"]

    A, B = (s_yeo, t_yeo) if YEO7.index(s_yeo) <= YEO7.index(t_yeo) else (t_yeo, s_yeo)
    _, _, denom, ptype = possible_edges_for_pair(A, B, yeo_counts)

    seed_ids.append(s_id)
    target_ids.append(t_id)
    seed_yeo_out.append(s_yeo)
    target_yeo_out.append(t_yeo)
    pair_a.append(A)
    pair_b.append(B)
    possible_edges.append(denom)
    pair_type_list.append(ptype)

rowlevel_df = pd.DataFrame({
    "Study": df[study_c].astype(str).values if study_c in df.columns else ["Palhano-Fontes"] * len(df),
    "Contrast": df[con_c].astype(str).values,
    "Seed_Node_ID": seed_ids,
    "Target_Node_ID": target_ids,
    "Seed_X": df[seed_x_c].values,
    "Seed_Y": df[seed_y_c].values,
    "Seed_Z": df[seed_z_c].values,
    "Seed_Radius_mm": df[seed_r_c].values,
    "Target_X": df[targ_x_c].values,
    "Target_Y": df[targ_y_c].values,
    "Target_Z": df[targ_z_c].values,
    "Target_Radius_mm": df[targ_r_c].values,
    "Seed_Yeo7": seed_yeo_out,
    "Target_Yeo7": target_yeo_out,
    "Yeo_Pair_A": pair_a,
    "Yeo_Pair_B": pair_b,
    "Direction": df[dir_c].astype(str).values,
    "Possible_Edges": possible_edges,
    "Pair_Type": pair_type_list,
})

# =========================
# Save
# =========================
with pd.ExcelWriter(OUT_XLSX, engine="openpyxl") as writer:
    node_df.to_excel(writer, sheet_name="SphereNode_to_Yeo7_mapping", index=False)
    yeo_counts_df.to_excel(writer, sheet_name="Yeo_counts", index=False)
    master_df.to_excel(writer, sheet_name="Master_possible_edges", index=False)
    rowlevel_df.to_excel(writer, sheet_name="Rowlevel_denominators", index=False)

print("✅ Saved:", OUT_XLSX)
print("\nYeo counts:")
print(yeo_counts_df)
print("\nMaster possible-edge table:")
print(master_df.head(15))

✅ Saved: /Users/maximegodart/Desktop/CBMA_coordinate_extract/CONNECTIVITY/NEW ANALYSIS DATA/possible_edge_outputs/PalhanoFontes_2015_possible_edges_master.xlsx

Yeo counts:
               Yeo7  N_Original_Nodes
0            Visual                 0
1       Somatomotor                 0
2   DorsalAttention                 0
3  VentralAttention                 0
4            Limbic                 0
5    Frontoparietal                 0
6           Default                 3

Master possible-edge table:
                  Study                      Atlas       Yeo_Pair_A  \
0   Palhano-Fontes 2015  Study-defined MNI spheres           Visual   
1   Palhano-Fontes 2015  Study-defined MNI spheres           Visual   
2   Palhano-Fontes 2015  Study-defined MNI spheres           Visual   
3   Palhano-Fontes 2015  Study-defined MNI spheres           Visual   
4   Palhano-Fontes 2015  Study-defined MNI spheres           Visual   
5   Palhano-Fontes 2015  Study-defined MNI spheres           Visual 

In [10]:
# ============================================
# PASQUINI 2020: NETWORK-COORDINATE SPHERES -> YEO-7 POSSIBLE EDGES
# Creates:
#   1) Pasquini network-to-Yeo mapping from union-of-spheres overlap
#   2) Yeo counts based on original Pasquini networks
#   3) master table with one row per Yeo pair + denominator
#   4) row-level table for reported edges
# ============================================

import os
import re
import numpy as np
import pandas as pd
import nibabel as nib

# =========================
# Paths
# =========================
BASE_DIR  = "/Users/maximegodart/Desktop/CBMA_coordinate_extract/CONNECTIVITY/NEW ANALYSIS DATA"
ATLAS_DIR = os.path.join(BASE_DIR, "Atlases")

COORD_XLSX = os.path.join(BASE_DIR, "Pasquini_2020_Network_Coordinates.xlsx")
EDGES_XLSX = os.path.join(BASE_DIR, "Pasquini_2020_significant edges_FINAL.xlsx")
YEO_NII    = os.path.join(ATLAS_DIR, "Yeo2011_7Networks_MNI152_FreeSurferConformed1mm.nii")

if not os.path.exists(YEO_NII):
    raise FileNotFoundError(f"Could not find Yeo atlas: {YEO_NII}")

OUT_DIR  = os.path.join(BASE_DIR, "possible_edge_outputs")
os.makedirs(OUT_DIR, exist_ok=True)
OUT_XLSX = os.path.join(OUT_DIR, "Pasquini_2020_possible_edges_master.xlsx")

RADIUS_MM = 10.0

YEO7 = [
    "Visual",
    "Somatomotor",
    "DorsalAttention",
    "VentralAttention",
    "Limbic",
    "Frontoparietal",
    "Default",
]

# =========================
# Helpers
# =========================
def clean_ws(s):
    return re.sub(r"\s+", " ", str(s).strip())

def norm_key(s):
    return re.sub(r"[^a-z0-9]+", "", clean_ws(s).lower())

def dice(a, b):
    inter = np.count_nonzero(a & b)
    na = np.count_nonzero(a)
    nb = np.count_nonzero(b)
    return 0.0 if (na + nb) == 0 else (2.0 * inter) / (na + nb)

def sphere_mask_on_grid(ref_img: nib.Nifti1Image, center_xyz_mm, radius_mm=10.0):
    """Boolean sphere mask in voxel grid of ref_img."""
    shape = ref_img.shape[:3]
    aff = ref_img.affine
    inv = np.linalg.inv(aff)

    cx, cy, cz = nib.affines.apply_affine(inv, np.array(center_xyz_mm, dtype=float))
    cx, cy, cz = float(cx), float(cy), float(cz)

    vx = float(np.linalg.norm(aff[:3, 0]))
    vy = float(np.linalg.norm(aff[:3, 1]))
    vz = float(np.linalg.norm(aff[:3, 2]))

    rx = int(np.ceil(radius_mm / vx))
    ry = int(np.ceil(radius_mm / vy))
    rz = int(np.ceil(radius_mm / vz))

    x0 = max(0, int(np.floor(cx)) - rx)
    x1 = min(shape[0], int(np.floor(cx)) + rx + 1)
    y0 = max(0, int(np.floor(cy)) - ry)
    y1 = min(shape[1], int(np.floor(cy)) + ry + 1)
    z0 = max(0, int(np.floor(cz)) - rz)
    z1 = min(shape[2], int(np.floor(cz)) + rz + 1)

    xs = np.arange(x0, x1)
    ys = np.arange(y0, y1)
    zs = np.arange(z0, z1)
    X, Y, Z = np.meshgrid(xs, ys, zs, indexing="ij")

    dx = (X - cx) * vx
    dy = (Y - cy) * vy
    dz = (Z - cz) * vz
    local = (dx * dx + dy * dy + dz * dz) <= (radius_mm ** 2)

    mask = np.zeros(shape, dtype=bool)
    mask[x0:x1, y0:y1, z0:z1] = local
    return mask

def get_yeo_masks(yeo_img):
    yd = yeo_img.get_fdata()
    if yd.ndim == 4 and yd.shape[-1] == 1:
        yd = np.squeeze(yd, axis=-1)

    yeo = yd.astype(int)
    yeo_ids = sorted([i for i in np.unique(yeo) if i != 0])[:7]
    yeo_id_to_name = {yeo_ids[i]: YEO7[i] for i in range(min(7, len(yeo_ids)))}

    masks = {}
    for yid in yeo_ids:
        nm = yeo_id_to_name.get(yid)
        if nm:
            masks[nm] = (yeo == yid)
    return masks

def best_yeo_for_mask(mask, yeo_masks):
    best_name = None
    best_d = -1.0
    best_inter = -1
    for nm, ym in yeo_masks.items():
        inter = np.count_nonzero(mask & ym)
        d = dice(mask, ym)
        if (d > best_d) or (np.isclose(d, best_d) and inter > best_inter):
            best_name = nm
            best_d = float(d)
            best_inter = int(inter)
    return best_name, best_d, best_inter

def possible_edges_for_pair(a, b, yeo_counts):
    nA = int(yeo_counts.get(a, 0))
    nB = int(yeo_counts.get(b, 0))
    if a == b:
        return nA, nB, nA * (nA - 1) // 2, "within"
    return nA, nB, nA * nB, "between"

# =========================
# Load coordinate table
# =========================
coords = pd.read_excel(COORD_XLSX)
coords.columns = coords.columns.astype(str).str.replace("\u00a0", " ", regex=False).str.strip()

rename_map = {}
for c in coords.columns:
    if norm_key(c) == "x":
        rename_map[c] = "x"
    elif norm_key(c) == "y":
        rename_map[c] = "y"
    elif norm_key(c) == "z":
        rename_map[c] = "z"
    elif norm_key(c) == "network":
        rename_map[c] = "Network"
    elif norm_key(c) == "region":
        rename_map[c] = "Region"
coords = coords.rename(columns=rename_map)

required_coord_cols = {"Network", "x", "y", "z"}
missing = required_coord_cols - set(coords.columns)
if missing:
    raise ValueError(f"Coordinate file missing columns: {missing}. Found: {list(coords.columns)}")

coords["Network"] = coords["Network"].astype(str).map(clean_ws)
if "Region" in coords.columns:
    coords["Region"] = coords["Region"].astype(str).map(clean_ws)

# =========================
# Load Yeo atlas + masks
# =========================
yeo_img = nib.load(YEO_NII)
yeo_masks = get_yeo_masks(yeo_img)

# =========================
# Build original network masks from 10mm spheres
# =========================
network_masks = {}
network_roi_counts = {}

for network in coords["Network"].dropna().unique():
    sub = coords[coords["Network"] == network].copy()
    mask = np.zeros(yeo_img.shape[:3], dtype=bool)

    for _, r in sub.iterrows():
        xyz = (float(r["x"]), float(r["y"]), float(r["z"]))
        mask |= sphere_mask_on_grid(yeo_img, xyz, radius_mm=RADIUS_MM)

    if np.count_nonzero(mask) == 0:
        raise ValueError(f"Network {network!r} produced an empty sphere mask.")

    network_masks[network] = mask
    network_roi_counts[network] = int(len(sub))

# =========================
# Map Pasquini networks -> Yeo-7 via Dice
# =========================
network_to_yeo = {}
network_to_dice = {}
network_to_inter = {}

for network, mask in network_masks.items():
    best, best_d, best_inter = best_yeo_for_mask(mask, yeo_masks)
    if best not in YEO7:
        raise ValueError(f"Network {network!r} mapped to invalid Yeo-7 label: {best!r}")
    network_to_yeo[network] = best
    network_to_dice[network] = best_d
    network_to_inter[network] = best_inter

mapping_rows = []
for network in sorted(network_to_yeo.keys()):
    mapping_rows.append({
        "Pasquini_Network": network,
        "Yeo7": network_to_yeo[network],
        "Dice": network_to_dice[network],
        "Intersect_Voxels": network_to_inter[network],
        "N_ROIs_in_Network": network_roi_counts[network],
    })
mapping_df = pd.DataFrame(mapping_rows)

# =========================
# Yeo counts from original Pasquini networks
# =========================
yeo_counts = (
    mapping_df.groupby("Yeo7", as_index=False)["Pasquini_Network"]
    .nunique()
    .rename(columns={"Pasquini_Network": "N_Pasquini_Networks"})
)
yeo_counts = yeo_counts.set_index("Yeo7")["N_Pasquini_Networks"].to_dict()
yeo_counts = {k: int(yeo_counts.get(k, 0)) for k in YEO7}

yeo_counts_df = pd.DataFrame({
    "Yeo7": YEO7,
    "N_Pasquini_Networks": [yeo_counts[k] for k in YEO7]
})

# =========================
# Master table: one row per Yeo pair
# =========================
master_rows = []
for i, a in enumerate(YEO7):
    for b in YEO7[i:]:
        nA, nB, possible, pair_type = possible_edges_for_pair(a, b, yeo_counts)
        master_rows.append({
            "Study": "Pasquini 2020",
            "Atlas": "Study-defined network coordinates",
            "Yeo_Pair_A": a,
            "Yeo_Pair_B": b,
            "Regions_in_A": nA,
            "Regions_in_B": nB,
            "Possible_Edges": possible,
            "Pair_Type": pair_type,
        })
master_df = pd.DataFrame(master_rows)

# =========================
# Row-level table for reported edges
# =========================
edges = pd.read_excel(EDGES_XLSX)
edges.columns = edges.columns.astype(str).str.replace("\u00a0", " ", regex=False).str.strip()

required_edge_cols = {"Study", "Contrast", "Seed", "Target", "Direction"}
missing = required_edge_cols - set(edges.columns)
if missing:
    raise ValueError(f"Edge file missing columns: {missing}. Found: {list(edges.columns)}")

def require_network_to_yeo(label, row_idx, colname):
    raw = clean_ws(label)

    if raw in network_to_yeo:
        return raw, network_to_yeo[raw]

    raw_norm = norm_key(raw)
    for k in network_to_yeo:
        if norm_key(k) == raw_norm:
            return k, network_to_yeo[k]

    hints = [k for k in network_to_yeo if raw_norm in norm_key(k) or norm_key(k) in raw_norm][:10]
    hint_txt = ""
    if hints:
        hint_txt = "\nClosest matches:\n  - " + "\n  - ".join(hints)

    raise ValueError(
        f"[Mapping error] Row {row_idx}: could not map {colname} label {raw!r} "
        f"to a network in Pasquini_2020_Network_Coordinates.{hint_txt}"
    )

seed_network_exact = []
target_network_exact = []
seed_yeo = []
target_yeo = []
pair_a = []
pair_b = []
possible_edges = []
pair_type_list = []

for i, r in edges.iterrows():
    s_net, s_yeo = require_network_to_yeo(r["Seed"], i, "Seed")
    t_net, t_yeo = require_network_to_yeo(r["Target"], i, "Target")

    A, B = (s_yeo, t_yeo) if YEO7.index(s_yeo) <= YEO7.index(t_yeo) else (t_yeo, s_yeo)
    _, _, denom, ptype = possible_edges_for_pair(A, B, yeo_counts)

    seed_network_exact.append(s_net)
    target_network_exact.append(t_net)
    seed_yeo.append(s_yeo)
    target_yeo.append(t_yeo)
    pair_a.append(A)
    pair_b.append(B)
    possible_edges.append(denom)
    pair_type_list.append(ptype)

rowlevel_df = pd.DataFrame({
    "Study": edges["Study"].astype(str).values,
    "Contrast": edges["Contrast"].astype(str).values,
    "Seed_Original": edges["Seed"].astype(str).values,
    "Target_Original": edges["Target"].astype(str).values,
    "Seed_Pasquini_Network": seed_network_exact,
    "Target_Pasquini_Network": target_network_exact,
    "Seed_Yeo7": seed_yeo,
    "Target_Yeo7": target_yeo,
    "Yeo_Pair_A": pair_a,
    "Yeo_Pair_B": pair_b,
    "Direction": edges["Direction"].astype(str).values,
    "Possible_Edges": possible_edges,
    "Pair_Type": pair_type_list,
})

# =========================
# Save
# =========================
with pd.ExcelWriter(OUT_XLSX, engine="openpyxl") as writer:
    mapping_df.to_excel(writer, sheet_name="Pasquini_to_Yeo7_mapping", index=False)
    coords.to_excel(writer, sheet_name="Network_coordinates", index=False)
    yeo_counts_df.to_excel(writer, sheet_name="Yeo_counts", index=False)
    master_df.to_excel(writer, sheet_name="Master_possible_edges", index=False)
    rowlevel_df.to_excel(writer, sheet_name="Rowlevel_denominators", index=False)

print("✅ Saved:", OUT_XLSX)
print("\nPasquini -> Yeo mapping:")
print(mapping_df)
print("\nYeo counts:")
print(yeo_counts_df)
print("\nMaster possible-edge table:")
print(master_df.head(15))

✅ Saved: /Users/maximegodart/Desktop/CBMA_coordinate_extract/CONNECTIVITY/NEW ANALYSIS DATA/possible_edge_outputs/Pasquini_2020_possible_edges_master.xlsx

Pasquini -> Yeo mapping:
  Pasquini_Network              Yeo7      Dice  Intersect_Voxels  \
0     Default Mode           Default  0.087163              6620   
1         Salience  VentralAttention  0.108397              3872   
2     Sensorimotor       Somatomotor  0.047038              1800   
3           Visual            Visual  0.056758              2101   

   N_ROIs_in_Network  
0                  4  
1                  3  
2                  2  
3                  2  

Yeo counts:
               Yeo7  N_Pasquini_Networks
0            Visual                    1
1       Somatomotor                    1
2   DorsalAttention                    0
3  VentralAttention                    1
4            Limbic                    0
5    Frontoparietal                    0
6           Default                    1

Master possible-edge 

In [11]:
# ============================================
# ROSEMAN 2014: SMITH RSN10 -> YEO-7 POSSIBLE EDGES
# Creates:
#   1) Smith RSN10 component-to-Yeo mapping from thresholded atlas overlap
#   2) Yeo counts based on original Smith RSN10 components
#   3) master table with one row per Yeo pair + denominator
#   4) row-level table for reported Roseman edges
# ============================================

import os
import re
import numpy as np
import pandas as pd
import nibabel as nib
from nilearn.image import resample_to_img

# =========================
# Paths
# =========================
BASE_DIR  = "/Users/maximegodart/Desktop/CBMA_coordinate_extract/CONNECTIVITY/NEW ANALYSIS DATA"
ATLAS_DIR = os.path.join(BASE_DIR, "Atlases")

ROSEMAN_XLSX = os.path.join(BASE_DIR, "Roseman_2014_significant_edges_FINAL.xlsx")
SMITH_NII    = os.path.join(ATLAS_DIR, "PNAS_Smith09_rsn10.nii.gz")
SMITH_LABELS = os.path.join(ATLAS_DIR, "SMITH09_RSN10_labels.txt")
YEO_NII      = os.path.join(ATLAS_DIR, "Yeo2011_7Networks_MNI152_FreeSurferConformed1mm.nii")

if not os.path.exists(SMITH_NII):
    raise FileNotFoundError(f"Could not find Smith atlas: {SMITH_NII}")
if not os.path.exists(SMITH_LABELS):
    raise FileNotFoundError(f"Could not find Smith labels: {SMITH_LABELS}")
if not os.path.exists(YEO_NII):
    raise FileNotFoundError(f"Could not find Yeo atlas: {YEO_NII}")

OUT_DIR = os.path.join(BASE_DIR, "possible_edge_outputs")
os.makedirs(OUT_DIR, exist_ok=True)
OUT_XLSX = os.path.join(OUT_DIR, "Roseman_2014_possible_edges_master.xlsx")

YEO7 = ["Visual","Somatomotor","DorsalAttention","VentralAttention","Limbic","Frontoparietal","Default"]

SMITH_Z_THRESHOLD = 2.3

# =========================
# Helpers
# =========================
def clean_ws(s):
    return re.sub(r"\s+", " ", str(s).strip())

def norm(s):
    s = clean_ws(s).lower()
    s = re.sub(r"[^a-z0-9]+", "", s)
    return s

def dice(a, b):
    inter = np.count_nonzero(a & b)
    na = np.count_nonzero(a)
    nb = np.count_nonzero(b)
    return 0.0 if (na + nb) == 0 else (2.0 * inter) / (na + nb)

def possible_edges_for_pair(a, b, yeo_counts):
    nA = int(yeo_counts.get(a, 0))
    nB = int(yeo_counts.get(b, 0))
    if a == b:
        return nA, nB, nA * (nA - 1) // 2, "within"
    return nA, nB, nA * nB, "between"

# =========================
# Load Smith labels
# =========================
def load_smith_labels(labels_path):
    id_to_label = {}
    with open(labels_path, "r", encoding="utf-8", errors="ignore") as f:
        for ln in f:
            ln = ln.strip()
            if not ln:
                continue
            m = re.match(r"^(\d+)\s+(.+)$", ln)
            if not m:
                continue
            rid = int(m.group(1))
            lab = clean_ws(m.group(2))
            id_to_label[rid] = lab
    label_to_id = {norm(v): k for k, v in id_to_label.items()}
    return id_to_label, label_to_id

SMITH_ALIAS = {
    "cerebellum": "Cerebellum",
    "auditory": "Auditory",
    "aud": "Auditory",
    "sensorimotor": "Sensorimotor",
    "sm": "Sensorimotor",
    "dmn": "Default mode network",
    "defaultmodenetwork": "Default mode network",
    "executive": "Executive control",
    "executivecontrol": "Executive control",
    "ecn": "Executive control",
    "lfp": "Left frontoparietal",
    "leftfrontoparietal": "Left frontoparietal",
    "frontoparietal1": "Left frontoparietal",
    "rfp": "Right frontoparietal",
    "rightfrontoparietal": "Right frontoparietal",
    "frontoparietal2": "Right frontoparietal",
    "visual1": "Visual medial",
    "visual2": "Visual occipital pole",
    "visual3": "Visual lateral",
    "vism": "Visual medial",
    "viso": "Visual occipital pole",
    "visl": "Visual lateral",
    "dan": "Executive control",
    "dan2": "Executive control",
    "dmn2": "Default mode network",
}

def canonical_label(x, row_idx, colname, study_name, id_to_label):
    raw = clean_ws(x)
    k = norm(raw)

    for rid, lab in id_to_label.items():
        if norm(lab) == k:
            return lab

    if k in SMITH_ALIAS:
        canon = SMITH_ALIAS[k]
        for rid, lab in id_to_label.items():
            if norm(lab) == norm(canon):
                return lab

    hints = [lab for rid, lab in id_to_label.items() if k in norm(lab) or norm(lab) in k][:10]
    hint_txt = "\nClosest matches:\n  - " + "\n  - ".join(hints) if hints else ""
    raise ValueError(
        f"[{study_name}] Row {row_idx}: could not map '{colname}' label to Smith RSN10.\n"
        f"  Raw value: {raw!r}\n"
        f"  Normalized: {k!r}{hint_txt}"
    )

# =========================
# Smith -> Yeo-7
# =========================
def compute_smith_to_yeo7(smith_nii_path, yeo_nii_path, zthr=2.3):
    smith_img = nib.load(smith_nii_path)
    yeo_img   = nib.load(yeo_nii_path)

    smith = smith_img.get_fdata()
    if smith.ndim != 4:
        raise ValueError(f"Expected Smith atlas to be 4D. Got shape: {smith.shape}")
    if smith.shape[-1] < 10:
        raise ValueError(f"Expected >=10 Smith components. Got: {smith.shape[-1]}")

    yd = yeo_img.get_fdata()
    if yd.ndim == 4 and yd.shape[-1] == 1:
        yeo_img = nib.Nifti1Image(np.squeeze(yd, axis=-1), affine=yeo_img.affine)

    yeo_rs = resample_to_img(yeo_img, smith_img, interpolation="nearest")
    yeo = yeo_rs.get_fdata().astype(int)

    yeo_ids = sorted([i for i in np.unique(yeo) if i != 0])[:7]
    yeo_id_to_name = {yeo_ids[i]: YEO7[i] for i in range(min(7, len(yeo_ids)))}

    smith_to_yeo = {}
    smith_to_dice = {}
    mapping_rows = []

    for k in range(10):
        comp = smith[..., k]
        comp_mask = comp > zthr

        if np.count_nonzero(comp_mask) == 0:
            raise ValueError(
                f"Smith component {k+1} has zero voxels above threshold {zthr}. "
                f"Try lowering SMITH_Z_THRESHOLD."
            )

        best_name, best_d, best_inter = None, -1.0, -1
        for yid in yeo_ids:
            ym = (yeo == yid)
            inter = np.count_nonzero(comp_mask & ym)
            d = dice(comp_mask, ym)
            if (d > best_d) or (np.isclose(d, best_d) and inter > best_inter):
                best_d, best_inter = float(d), int(inter)
                best_name = yeo_id_to_name.get(yid)

        comp_id = k + 1
        smith_to_yeo[comp_id] = best_name
        smith_to_dice[comp_id] = best_d
        mapping_rows.append({
            "Smith_Component": comp_id,
            "Best_Yeo7": best_name,
            "Dice": best_d,
            "Intersect_Voxels": best_inter,
            "Component_Voxels": int(np.count_nonzero(comp_mask)),
        })

    overlap_df = pd.DataFrame(mapping_rows)
    return smith_to_yeo, smith_to_dice, overlap_df

# =========================
# Build mapping and Yeo counts
# =========================
id_to_label, label_to_id = load_smith_labels(SMITH_LABELS)
smith_to_yeo, smith_to_dice, overlap_df = compute_smith_to_yeo7(SMITH_NII, YEO_NII, zthr=SMITH_Z_THRESHOLD)

mapping_rows = []
for comp_id in sorted(id_to_label.keys()):
    mapping_rows.append({
        "Smith_Component": comp_id,
        "Smith_Label": id_to_label[comp_id],
        "Yeo7": smith_to_yeo.get(comp_id),
        "Dice": smith_to_dice.get(comp_id),
    })
mapping_df = pd.DataFrame(mapping_rows)

yeo_counts = (
    mapping_df.groupby("Yeo7", as_index=False)["Smith_Component"]
    .nunique()
    .rename(columns={"Smith_Component": "N_Smith_Components"})
)
yeo_counts = yeo_counts.set_index("Yeo7")["N_Smith_Components"].to_dict()
yeo_counts = {k: int(yeo_counts.get(k, 0)) for k in YEO7}

yeo_counts_df = pd.DataFrame({
    "Yeo7": YEO7,
    "N_Smith_Components": [yeo_counts[k] for k in YEO7]
})

# =========================
# Master table
# =========================
master_rows = []
for i, a in enumerate(YEO7):
    for b in YEO7[i:]:
        nA, nB, possible, pair_type = possible_edges_for_pair(a, b, yeo_counts)
        master_rows.append({
            "Study": "Roseman 2014",
            "Atlas": "Smith RSN10",
            "Yeo_Pair_A": a,
            "Yeo_Pair_B": b,
            "Regions_in_A": nA,
            "Regions_in_B": nB,
            "Possible_Edges": possible,
            "Pair_Type": pair_type,
        })
master_df = pd.DataFrame(master_rows)

# =========================
# Row-level table for Roseman
# =========================
df = pd.read_excel(ROSEMAN_XLSX)
df.columns = df.columns.astype(str).str.replace("\u00a0", " ", regex=False).str.strip()

seed_c, targ_c, dir_c = "Seed", "Target", "Direction"
con_c = "Contrast"
net_c = "Network"

if net_c in df.columns:
    net_norm = (
        df[net_c].astype(str).str.strip().str.lower()
        .str.replace(r"[^a-z0-9]+", "", regex=True)
    )
    sub = df[net_norm.isin(["smith2009", "smith", "smithrsn10"])].copy()
    if len(sub) == 0:
        sub = df.copy()
else:
    sub = df.copy()

seed_label_exact = []
targ_label_exact = []
seed_comp = []
targ_comp = []
seed_yeo = []
targ_yeo = []
pair_a = []
pair_b = []
possible_edges = []
pair_type_list = []

for i, r in sub.iterrows():
    s_lab = canonical_label(r[seed_c], i, "Seed", "Roseman", id_to_label)
    t_lab = canonical_label(r[targ_c], i, "Target", "Roseman", id_to_label)

    s_id = label_to_id[norm(s_lab)]
    t_id = label_to_id[norm(t_lab)]
    s_yeo = smith_to_yeo.get(s_id)
    t_yeo = smith_to_yeo.get(t_id)

    if s_yeo not in YEO7:
        raise ValueError(f"[Roseman] Row {i}: Seed {s_lab!r} mapped to invalid Yeo value: {s_yeo!r}")
    if t_yeo not in YEO7:
        raise ValueError(f"[Roseman] Row {i}: Target {t_lab!r} mapped to invalid Yeo value: {t_yeo!r}")

    A, B = (s_yeo, t_yeo) if YEO7.index(s_yeo) <= YEO7.index(t_yeo) else (t_yeo, s_yeo)
    _, _, denom, ptype = possible_edges_for_pair(A, B, yeo_counts)

    seed_label_exact.append(s_lab)
    targ_label_exact.append(t_lab)
    seed_comp.append(s_id)
    targ_comp.append(t_id)
    seed_yeo.append(s_yeo)
    targ_yeo.append(t_yeo)
    pair_a.append(A)
    pair_b.append(B)
    possible_edges.append(denom)
    pair_type_list.append(ptype)

rowlevel_df = pd.DataFrame({
    "Study": "Roseman",
    "Contrast": sub[con_c].astype(str).values,
    "Seed_Original": sub[seed_c].astype(str).values,
    "Target_Original": sub[targ_c].astype(str).values,
    "Seed_Smith_Label": seed_label_exact,
    "Target_Smith_Label": targ_label_exact,
    "Seed_Smith_Component": seed_comp,
    "Target_Smith_Component": targ_comp,
    "Seed_Yeo7": seed_yeo,
    "Target_Yeo7": targ_yeo,
    "Yeo_Pair_A": pair_a,
    "Yeo_Pair_B": pair_b,
    "Direction": sub[dir_c].astype(str).values,
    "Possible_Edges": possible_edges,
    "Pair_Type": pair_type_list,
})

# =========================
# Save
# =========================
with pd.ExcelWriter(OUT_XLSX, engine="openpyxl") as writer:
    mapping_df.to_excel(writer, sheet_name="Smith_to_Yeo7_mapping", index=False)
    overlap_df.to_excel(writer, sheet_name="Smith_overlap_debug", index=False)
    yeo_counts_df.to_excel(writer, sheet_name="Yeo_counts", index=False)
    master_df.to_excel(writer, sheet_name="Master_possible_edges", index=False)
    rowlevel_df.to_excel(writer, sheet_name="Rowlevel_denominators", index=False)

print("✅ Saved:", OUT_XLSX)
print("\nSmith -> Yeo mapping:")
print(mapping_df)
print("\nYeo counts:")
print(yeo_counts_df)
print("\nMaster possible-edge table:")
print(master_df.head(15))

✅ Saved: /Users/maximegodart/Desktop/CBMA_coordinate_extract/CONNECTIVITY/NEW ANALYSIS DATA/possible_edge_outputs/Roseman_2014_possible_edges_master.xlsx

Smith -> Yeo mapping:
   Smith_Component            Smith_Label              Yeo7      Dice
0                1          Visual medial            Visual  0.278647
1                2  Visual occipital pole            Visual  0.198226
2                3         Visual lateral            Visual  0.234508
3                4   Default mode network           Default  0.342841
4                5             Cerebellum            Visual  0.062025
5                6           Sensorimotor       Somatomotor  0.244239
6                7               Auditory  VentralAttention  0.217414
7                8      Executive control    Frontoparietal  0.146063
8                9    Left frontoparietal    Frontoparietal  0.219659
9               10   Right frontoparietal    Frontoparietal  0.183292

Yeo counts:
               Yeo7  N_Smith_Components


In [12]:
# ============================================
# SMIGIELSKI 2019: HARVARD-OXFORD -> YEO-7 POSSIBLE EDGES
# Creates:
#   1) Harvard-Oxford parcel-to-Yeo mapping from atlas overlap
#   2) Yeo counts based on original Harvard-Oxford parcels
#   3) master table with one row per Yeo pair + denominator
#   4) row-level table for reported Smigielski edges
# ============================================

import os
import re
import numpy as np
import pandas as pd
import nibabel as nib
from nilearn.image import resample_to_img

# =========================
# Paths
# =========================
BASE_DIR  = "/Users/maximegodart/Desktop/CBMA_coordinate_extract/CONNECTIVITY/NEW ANALYSIS DATA"
ATLAS_DIR = os.path.join(BASE_DIR, "Atlases")

SMIGIELSKI_XLSX = os.path.join(BASE_DIR, "Smigielski_2019_significant_edges_FINAL.xlsx")

HO_NII  = os.path.join(ATLAS_DIR, "conn_atlas_harvardoxford.nii")
HO_TXT  = os.path.join(ATLAS_DIR, "conn_atlas_harvardoxford.txt")
YEO_NII = os.path.join(ATLAS_DIR, "Yeo2011_7Networks_MNI152_FreeSurferConformed1mm.nii")

if not os.path.exists(HO_NII):
    raise FileNotFoundError(f"Could not find Harvard-Oxford atlas: {HO_NII}")
if not os.path.exists(HO_TXT):
    raise FileNotFoundError(f"Could not find Harvard-Oxford labels: {HO_TXT}")
if not os.path.exists(YEO_NII):
    raise FileNotFoundError(f"Could not find Yeo atlas: {YEO_NII}")

OUT_DIR = os.path.join(BASE_DIR, "possible_edge_outputs")
os.makedirs(OUT_DIR, exist_ok=True)

OUT_XLSX = os.path.join(OUT_DIR, "Smigielski_2019_possible_edges_master.xlsx")

YEO7 = ["Visual","Somatomotor","DorsalAttention","VentralAttention","Limbic","Frontoparietal","Default"]

# =========================
# Helpers
# =========================
def clean_ws(s):
    return re.sub(r"\s+", " ", str(s).strip())

def dice(a, b):
    inter = np.count_nonzero(a & b)
    na = np.count_nonzero(a)
    nb = np.count_nonzero(b)
    return 0.0 if (na + nb) == 0 else (2.0 * inter) / (na + nb)

def load_ho_label_to_id(txt_path):
    labels = []
    with open(txt_path, "r", encoding="utf-8", errors="ignore") as f:
        for ln in f:
            ln = ln.strip()
            if ln:
                labels.append(clean_ws(ln))
    return {lab: i + 1 for i, lab in enumerate(labels)}

def compute_ho_to_yeo7(ho_nii_path, yeo_nii_path):
    ho_img  = nib.load(ho_nii_path)
    yeo_img = nib.load(yeo_nii_path)

    yd = yeo_img.get_fdata()
    if yd.ndim == 4 and yd.shape[-1] == 1:
        yeo_img = nib.Nifti1Image(np.squeeze(yd, axis=-1), affine=yeo_img.affine)

    yeo_rs = resample_to_img(yeo_img, ho_img, interpolation="nearest")
    ho  = ho_img.get_fdata().astype(int)
    yeo = yeo_rs.get_fdata().astype(int)

    ho_ids  = [i for i in np.unique(ho) if i != 0]
    yeo_ids = sorted([i for i in np.unique(yeo) if i != 0])[:7]
    yeo_id_to_name = {yeo_ids[i]: YEO7[i] for i in range(min(7, len(yeo_ids)))}

    ho_to_yeo = {}
    ho_to_dice = {}

    for hid in ho_ids:
        hm = (ho == hid)
        best_name, best_d, best_inter = None, -1.0, -1
        for yid in yeo_ids:
            ym = (yeo == yid)
            inter = np.count_nonzero(hm & ym)
            d = dice(hm, ym)
            if (d > best_d) or (np.isclose(d, best_d) and inter > best_inter):
                best_d, best_inter = float(d), int(inter)
                best_name = yeo_id_to_name.get(yid)
        ho_to_yeo[hid] = best_name
        ho_to_dice[hid] = best_d

    return ho_to_yeo, ho_to_dice

def require_map(label, ho_label_to_id, ho_to_yeo, row_idx, colname, study_tag):
    raw = label
    lab = clean_ws(raw)

    if lab not in ho_label_to_id:
        hints = [k for k in ho_label_to_id.keys() if lab.lower() in k.lower() or k.lower() in lab.lower()]
        hint_txt = ""
        if hints:
            hint_txt = "\nClosest matches:\n  - " + "\n  - ".join(hints[:10])
        raise ValueError(
            f"[{study_tag}] Row {row_idx}: '{colname}' not found in Harvard-Oxford labels.\n"
            f"  Raw value: {raw!r}\n"
            f"  Cleaned: {lab!r}{hint_txt}"
        )

    hid = ho_label_to_id[lab]
    yeo = ho_to_yeo.get(hid, None)
    if yeo not in YEO7:
        raise ValueError(
            f"[{study_tag}] Row {row_idx}: '{colname}' mapped HO id {hid} but got invalid Yeo-7 value: {yeo!r}\n"
            f"  Raw value: {raw!r}\n"
            f"  Cleaned: {lab!r}"
        )

    return hid, yeo

def compute_yeo_counts_from_ho(ho_label_to_id, ho_to_yeo):
    rows = []
    for lab, hid in ho_label_to_id.items():
        rows.append({
            "HO_ID": hid,
            "HO_Label": lab,
            "Yeo7": ho_to_yeo.get(hid, None),
        })

    atlas_df = pd.DataFrame(rows).sort_values("HO_ID").reset_index(drop=True)
    counts = atlas_df.groupby("Yeo7")["HO_ID"].nunique().to_dict()
    counts = {k: int(counts.get(k, 0)) for k in YEO7}
    return atlas_df, counts

def possible_edges_for_pair(a, b, yeo_counts):
    nA = int(yeo_counts.get(a, 0))
    nB = int(yeo_counts.get(b, 0))
    if a == b:
        return nA, nB, nA * (nA - 1) // 2, "within"
    return nA, nB, nA * nB, "between"

# =========================
# Build HO -> Yeo mapping + counts
# =========================
ho_label_to_id = load_ho_label_to_id(HO_TXT)
ho_to_yeo, ho_to_dice = compute_ho_to_yeo7(HO_NII, YEO_NII)
atlas_df, yeo_counts = compute_yeo_counts_from_ho(ho_label_to_id, ho_to_yeo)

mapping_df = atlas_df.copy()
mapping_df["Dice"] = mapping_df["HO_ID"].map(ho_to_dice)

yeo_counts_df = pd.DataFrame({
    "Yeo7": YEO7,
    "N_HO_Parcels": [yeo_counts[k] for k in YEO7]
})

# =========================
# Master table
# =========================
master_rows = []
for i, a in enumerate(YEO7):
    for b in YEO7[i:]:
        nA, nB, possible, pair_type = possible_edges_for_pair(a, b, yeo_counts)
        master_rows.append({
            "Study": "Smigielski 2019",
            "Atlas": "Harvard-Oxford",
            "Yeo_Pair_A": a,
            "Yeo_Pair_B": b,
            "Regions_in_A": nA,
            "Regions_in_B": nB,
            "Possible_Edges": possible,
            "Pair_Type": pair_type,
        })
master_df = pd.DataFrame(master_rows)

# =========================
# Row-level table for Smigielski
# =========================
df = pd.read_excel(SMIGIELSKI_XLSX)
df.columns = df.columns.astype(str).str.replace("\u00a0", " ", regex=False).str.strip()

seed_c, targ_c, dir_c, net_c, con_c = "Seed", "Target", "Direction", "Network", "Contrast"

sub = df[df[net_c].astype(str).str.strip().str.lower().eq("harvard-oxford")].copy()

seed_ids = []
targ_ids = []
seed_yeo = []
targ_yeo = []
pair_a = []
pair_b = []
possible_edges = []
pair_type_list = []

for i, r in sub.iterrows():
    s_id, s_yeo = require_map(r[seed_c], ho_label_to_id, ho_to_yeo, i, "Seed", "Smigielski")
    t_id, t_yeo = require_map(r[targ_c], ho_label_to_id, ho_to_yeo, i, "Target", "Smigielski")

    A, B = (s_yeo, t_yeo) if YEO7.index(s_yeo) <= YEO7.index(t_yeo) else (t_yeo, s_yeo)
    _, _, denom, ptype = possible_edges_for_pair(A, B, yeo_counts)

    seed_ids.append(s_id)
    targ_ids.append(t_id)
    seed_yeo.append(s_yeo)
    targ_yeo.append(t_yeo)
    pair_a.append(A)
    pair_b.append(B)
    possible_edges.append(denom)
    pair_type_list.append(ptype)

rowlevel_df = pd.DataFrame({
    "Study": "Smigielski",
    "Contrast": sub[con_c].astype(str).values,
    "Seed_Original": sub[seed_c].astype(str).values,
    "Target_Original": sub[targ_c].astype(str).values,
    "Seed_HO_ID": seed_ids,
    "Target_HO_ID": targ_ids,
    "Seed_Yeo7": seed_yeo,
    "Target_Yeo7": targ_yeo,
    "Yeo_Pair_A": pair_a,
    "Yeo_Pair_B": pair_b,
    "Direction": sub[dir_c].astype(str).values,
    "Possible_Edges": possible_edges,
    "Pair_Type": pair_type_list,
})

# =========================
# Save
# =========================
with pd.ExcelWriter(OUT_XLSX, engine="openpyxl") as writer:
    mapping_df.to_excel(writer, sheet_name="HO_to_Yeo7_mapping", index=False)
    yeo_counts_df.to_excel(writer, sheet_name="Yeo_counts", index=False)
    master_df.to_excel(writer, sheet_name="Master_possible_edges", index=False)
    rowlevel_df.to_excel(writer, sheet_name="Rowlevel_denominators", index=False)

print("✅ Saved:", OUT_XLSX)
print("\nYeo counts:")
print(yeo_counts_df)
print("\nMaster possible-edge table:")
print(master_df.head(15))

✅ Saved: /Users/maximegodart/Desktop/CBMA_coordinate_extract/CONNECTIVITY/NEW ANALYSIS DATA/possible_edge_outputs/Smigielski_2019_possible_edges_master.xlsx

Yeo counts:
               Yeo7  N_HO_Parcels
0            Visual            55
1       Somatomotor            19
2   DorsalAttention             8
3  VentralAttention             8
4            Limbic            20
5    Frontoparietal            10
6           Default            12

Master possible-edge table:
              Study           Atlas       Yeo_Pair_A        Yeo_Pair_B  \
0   Smigielski 2019  Harvard-Oxford           Visual            Visual   
1   Smigielski 2019  Harvard-Oxford           Visual       Somatomotor   
2   Smigielski 2019  Harvard-Oxford           Visual   DorsalAttention   
3   Smigielski 2019  Harvard-Oxford           Visual  VentralAttention   
4   Smigielski 2019  Harvard-Oxford           Visual            Limbic   
5   Smigielski 2019  Harvard-Oxford           Visual    Frontoparietal   
6   Smigiel

In [1]:
# ============================================
# GADDIS 2022: SMITH RSN10 -> YEO-7 POSSIBLE EDGES
# Creates:
#   1) Smith RSN10 component-to-Yeo mapping from thresholded atlas overlap
#   2) Yeo counts based on original Smith RSN10 components
#   3) master table with one row per Yeo pair + denominator
#   4) row-level table for reported Gaddis edges
# ============================================

import os
import re
import numpy as np
import pandas as pd
import nibabel as nib
from nilearn.image import resample_to_img

# =========================
# Paths
# =========================
BASE_DIR  = "/Users/maximegodart/Desktop/CBMA_coordinate_extract/CONNECTIVITY/NEW ANALYSIS DATA"
ATLAS_DIR = os.path.join(BASE_DIR, "Atlases")

GADDIS_XLSX  = os.path.join(BASE_DIR, "Gaddis_2022_significant_edges_FINAL.xlsx")
SMITH_NII    = os.path.join(ATLAS_DIR, "PNAS_Smith09_rsn10.nii.gz")
SMITH_LABELS = os.path.join(ATLAS_DIR, "SMITH09_RSN10_labels.txt")
YEO_NII      = os.path.join(ATLAS_DIR, "Yeo2011_7Networks_MNI152_FreeSurferConformed1mm.nii")

if not os.path.exists(SMITH_NII):
    raise FileNotFoundError(f"Could not find Smith atlas: {SMITH_NII}")
if not os.path.exists(SMITH_LABELS):
    raise FileNotFoundError(f"Could not find Smith labels: {SMITH_LABELS}")
if not os.path.exists(YEO_NII):
    raise FileNotFoundError(f"Could not find Yeo atlas: {YEO_NII}")

OUT_DIR = os.path.join(BASE_DIR, "possible_edge_outputs")
os.makedirs(OUT_DIR, exist_ok=True)
OUT_XLSX = os.path.join(OUT_DIR, "Gaddis_2022_possible_edges_master.xlsx")

YEO7 = ["Visual","Somatomotor","DorsalAttention","VentralAttention","Limbic","Frontoparietal","Default"]

# IMPORTANT: threshold Smith ICA maps before masks
SMITH_Z_THRESHOLD = 2.3

# =========================
# Helpers
# =========================
def clean_ws(s):
    return re.sub(r"\s+", " ", str(s).strip())

def norm(s):
    s = clean_ws(s).lower()
    s = re.sub(r"[^a-z0-9]+", "", s)
    return s

def dice(a, b):
    inter = np.count_nonzero(a & b)
    na = np.count_nonzero(a)
    nb = np.count_nonzero(b)
    return 0.0 if (na + nb) == 0 else (2.0 * inter) / (na + nb)

def possible_edges_for_pair(a, b, yeo_counts):
    nA = int(yeo_counts.get(a, 0))
    nB = int(yeo_counts.get(b, 0))
    if a == b:
        return nA, nB, nA * (nA - 1) // 2, "within"
    return nA, nB, nA * nB, "between"

# =========================
# Load Smith RSN10 labels
# =========================
def load_smith_labels(labels_path):
    id_to_label = {}
    with open(labels_path, "r", encoding="utf-8", errors="ignore") as f:
        for ln in f:
            ln = ln.strip()
            if not ln:
                continue
            m = re.match(r"^(\d+)\s+(.+)$", ln)
            if not m:
                continue
            rid = int(m.group(1))
            lab = clean_ws(m.group(2))
            id_to_label[rid] = lab
    label_to_id = {norm(v): k for k, v in id_to_label.items()}
    return id_to_label, label_to_id

SMITH_ALIAS = {
    "cerebellum": "Cerebellum",
    "cerebellar": "Cerebellum",
    "auditory": "Auditory",
    "aud": "Auditory",
    "sensorimotor": "Sensorimotor",
    "sm": "Sensorimotor",
    "dmn": "Default mode network",
    "defaultmode": "Default mode network",
    "defaultmodenetwork": "Default mode network",
    "executive": "Executive control",
    "executivecontrol": "Executive control",
    "ecn": "Executive control",
    "lfp": "Left frontoparietal",
    "leftfrontoparietal": "Left frontoparietal",
    "frontoparietal1": "Left frontoparietal",
    "rfp": "Right frontoparietal",
    "rightfrontoparietal": "Right frontoparietal",
    "frontoparietal2": "Right frontoparietal",
    "visual1": "Visual medial",
    "visual2": "Visual occipital pole",
    "visual3": "Visual lateral",
    "vism": "Visual medial",
    "viso": "Visual occipital pole",
    "visl": "Visual lateral",
    "visualmedial": "Visual medial",
    "visualoccipitalpole": "Visual occipital pole",
    "visuallateral": "Visual lateral",
    "dan": "Executive control",
    "dan2": "Executive control",
    "dmn2": "Default mode network",
}

def canonical_label(x, row_idx, colname, study_name, id_to_label):
    raw = clean_ws(x)
    k = norm(raw)

    for rid, lab in id_to_label.items():
        if norm(lab) == k:
            return lab

    if k in SMITH_ALIAS:
        canon = SMITH_ALIAS[k]
        for rid, lab in id_to_label.items():
            if norm(lab) == norm(canon):
                return lab

    hints = [lab for rid, lab in id_to_label.items() if k in norm(lab) or norm(lab) in k][:10]
    hint_txt = "\nClosest matches:\n  - " + "\n  - ".join(hints) if hints else ""
    raise ValueError(
        f"[{study_name}] Row {row_idx}: could not map '{colname}' label to Smith RSN10.\n"
        f"  Raw value: {raw!r}\n"
        f"  Normalized: {k!r}{hint_txt}"
    )

# =========================
# Smith RSN10 -> Yeo-7 via thresholded Dice
# =========================
def compute_smith_to_yeo7(smith_nii_path, yeo_nii_path, zthr=2.3):
    smith_img = nib.load(smith_nii_path)
    yeo_img   = nib.load(yeo_nii_path)

    smith = smith_img.get_fdata()
    if smith.ndim != 4:
        raise ValueError(f"Expected Smith atlas to be 4D. Got shape: {smith.shape}")
    if smith.shape[-1] < 10:
        raise ValueError(f"Expected >=10 Smith components. Got: {smith.shape[-1]}")

    yd = yeo_img.get_fdata()
    if yd.ndim == 4 and yd.shape[-1] == 1:
        yeo_img = nib.Nifti1Image(np.squeeze(yd, axis=-1), affine=yeo_img.affine)

    yeo_rs = resample_to_img(yeo_img, smith_img, interpolation="nearest")
    yeo = yeo_rs.get_fdata().astype(int)

    yeo_ids = sorted([i for i in np.unique(yeo) if i != 0])[:7]
    yeo_id_to_name = {yeo_ids[i]: YEO7[i] for i in range(min(7, len(yeo_ids)))}

    smith_to_yeo = {}
    smith_to_dice = {}
    mapping_rows = []

    for k in range(10):
        comp = smith[..., k]
        comp_mask = comp > zthr

        if np.count_nonzero(comp_mask) == 0:
            raise ValueError(
                f"Smith component {k+1} has zero voxels above threshold {zthr}. "
                f"Try lowering SMITH_Z_THRESHOLD."
            )

        best_name, best_d, best_inter = None, -1.0, -1
        for yid in yeo_ids:
            ym = (yeo == yid)
            inter = np.count_nonzero(comp_mask & ym)
            d = dice(comp_mask, ym)
            if (d > best_d) or (np.isclose(d, best_d) and inter > best_inter):
                best_d, best_inter = float(d), int(inter)
                best_name = yeo_id_to_name.get(yid)

        comp_id = k + 1
        smith_to_yeo[comp_id] = best_name
        smith_to_dice[comp_id] = best_d
        mapping_rows.append({
            "Smith_Component": comp_id,
            "Best_Yeo7": best_name,
            "Dice": best_d,
            "Intersect_Voxels": best_inter,
            "Component_Voxels": int(np.count_nonzero(comp_mask)),
        })

    mapping_df = pd.DataFrame(mapping_rows)
    return smith_to_yeo, smith_to_dice, mapping_df

# =========================
# Build mapping + Yeo counts
# =========================
id_to_label, label_to_id = load_smith_labels(SMITH_LABELS)
smith_to_yeo, smith_to_dice, overlap_df = compute_smith_to_yeo7(SMITH_NII, YEO_NII, zthr=SMITH_Z_THRESHOLD)

mapping_rows = []
for comp_id in sorted(id_to_label.keys()):
    mapping_rows.append({
        "Smith_Component": comp_id,
        "Smith_Label": id_to_label[comp_id],
        "Yeo7": smith_to_yeo.get(comp_id),
        "Dice": smith_to_dice.get(comp_id),
    })
mapping_df = pd.DataFrame(mapping_rows)

yeo_counts = (
    mapping_df.groupby("Yeo7", as_index=False)["Smith_Component"]
    .nunique()
    .rename(columns={"Smith_Component": "N_Smith_Components"})
)
yeo_counts = yeo_counts.set_index("Yeo7")["N_Smith_Components"].to_dict()
yeo_counts = {k: int(yeo_counts.get(k, 0)) for k in YEO7}

yeo_counts_df = pd.DataFrame({
    "Yeo7": YEO7,
    "N_Smith_Components": [yeo_counts[k] for k in YEO7]
})

# =========================
# Master table: one row per Yeo pair
# =========================
master_rows = []
for i, a in enumerate(YEO7):
    for b in YEO7[i:]:
        nA, nB, possible, pair_type = possible_edges_for_pair(a, b, yeo_counts)
        master_rows.append({
            "Study": "Gaddis 2022",
            "Atlas": "Smith RSN10",
            "Yeo_Pair_A": a,
            "Yeo_Pair_B": b,
            "Regions_in_A": nA,
            "Regions_in_B": nB,
            "Possible_Edges": possible,
            "Pair_Type": pair_type,
        })
master_df = pd.DataFrame(master_rows)

# =========================
# Row-level table for Gaddis edges
# =========================
df = pd.read_excel(GADDIS_XLSX)
df.columns = df.columns.astype(str).str.replace("\u00a0", " ", regex=False).str.strip()

seed_c, targ_c, dir_c = "Seed", "Target", "Direction"
con_c = "Contrast"
net_c = "Network"

if net_c in df.columns:
    net_norm = (
        df[net_c].astype(str).str.strip().str.lower()
        .str.replace(r"[^a-z0-9]+", "", regex=True)
    )
    sub = df[net_norm.isin(["smith2009", "smith", "smithrsn10"])].copy()
    if len(sub) == 0:
        sub = df.copy()
else:
    sub = df.copy()

seed_label_exact = []
targ_label_exact = []
seed_comp = []
targ_comp = []
seed_yeo = []
targ_yeo = []
pair_a = []
pair_b = []
possible_edges = []
pair_type_list = []

for i, r in sub.iterrows():
    s_lab = canonical_label(r[seed_c], i, "Seed", "Gaddis", id_to_label)
    t_lab = canonical_label(r[targ_c], i, "Target", "Gaddis", id_to_label)

    s_id = label_to_id[norm(s_lab)]
    t_id = label_to_id[norm(t_lab)]
    s_yeo = smith_to_yeo.get(s_id)
    t_yeo = smith_to_yeo.get(t_id)

    if s_yeo not in YEO7:
        raise ValueError(f"[Gaddis] Row {i}: Seed {s_lab!r} mapped to invalid Yeo value: {s_yeo!r}")
    if t_yeo not in YEO7:
        raise ValueError(f"[Gaddis] Row {i}: Target {t_lab!r} mapped to invalid Yeo value: {t_yeo!r}")

    A, B = (s_yeo, t_yeo) if YEO7.index(s_yeo) <= YEO7.index(t_yeo) else (t_yeo, s_yeo)
    _, _, denom, ptype = possible_edges_for_pair(A, B, yeo_counts)

    seed_label_exact.append(s_lab)
    targ_label_exact.append(t_lab)
    seed_comp.append(s_id)
    targ_comp.append(t_id)
    seed_yeo.append(s_yeo)
    targ_yeo.append(t_yeo)
    pair_a.append(A)
    pair_b.append(B)
    possible_edges.append(denom)
    pair_type_list.append(ptype)

rowlevel_df = pd.DataFrame({
    "Study": "Gaddis",
    "Contrast": sub[con_c].astype(str).values,
    "Seed_Original": sub[seed_c].astype(str).values,
    "Target_Original": sub[targ_c].astype(str).values,
    "Seed_Smith_Label": seed_label_exact,
    "Target_Smith_Label": targ_label_exact,
    "Seed_Smith_Component": seed_comp,
    "Target_Smith_Component": targ_comp,
    "Seed_Yeo7": seed_yeo,
    "Target_Yeo7": targ_yeo,
    "Yeo_Pair_A": pair_a,
    "Yeo_Pair_B": pair_b,
    "Direction": sub[dir_c].astype(str).values,
    "Possible_Edges": possible_edges,
    "Pair_Type": pair_type_list,
})

# =========================
# Save
# =========================
with pd.ExcelWriter(OUT_XLSX, engine="openpyxl") as writer:
    mapping_df.to_excel(writer, sheet_name="Smith_to_Yeo7_mapping", index=False)
    overlap_df.to_excel(writer, sheet_name="Smith_overlap_debug", index=False)
    yeo_counts_df.to_excel(writer, sheet_name="Yeo_counts", index=False)
    master_df.to_excel(writer, sheet_name="Master_possible_edges", index=False)
    rowlevel_df.to_excel(writer, sheet_name="Rowlevel_denominators", index=False)

print("✅ Saved:", OUT_XLSX)
print("\nSmith -> Yeo mapping:")
print(mapping_df)
print("\nYeo counts:")
print(yeo_counts_df)
print("\nMaster possible-edge table:")
print(master_df.head(15))

✅ Saved: /Users/maximegodart/Desktop/CBMA_coordinate_extract/CONNECTIVITY/NEW ANALYSIS DATA/possible_edge_outputs/Gaddis_2022_possible_edges_master.xlsx

Smith -> Yeo mapping:
   Smith_Component            Smith_Label              Yeo7      Dice
0                1          Visual medial            Visual  0.278647
1                2  Visual occipital pole            Visual  0.198226
2                3         Visual lateral            Visual  0.234508
3                4   Default mode network           Default  0.342841
4                5             Cerebellum            Visual  0.062025
5                6           Sensorimotor       Somatomotor  0.244239
6                7               Auditory  VentralAttention  0.217414
7                8      Executive control    Frontoparietal  0.146063
8                9    Left frontoparietal    Frontoparietal  0.219659
9               10   Right frontoparietal    Frontoparietal  0.183292

Yeo counts:
               Yeo7  N_Smith_Components
0

In [3]:
# ============================================
# BARRETT 2020: POWER 264 -> YEO-7 POSSIBLE EDGES
# Revised seed matching for labels like "Left Insula" / "Right Insula"
# ============================================

import os
import re
import numpy as np
import pandas as pd
import nibabel as nib
from nilearn.image import resample_to_img

BASE_DIR  = "/Users/maximegodart/Desktop/CBMA_coordinate_extract/CONNECTIVITY/NEW ANALYSIS DATA"
ATLAS_DIR = os.path.join(BASE_DIR, "Atlases")

BARRETT_XLSX = os.path.join(BASE_DIR, "Barrett_2020_significant_edges_FINAL.xlsx")

POWER_NII   = os.path.join(ATLAS_DIR, "power264MNI.nii.gz")
POWER_NAMES = os.path.join(ATLAS_DIR, "power264NodeNames.txt")
POWER_COMM_AFFIL = os.path.join(ATLAS_DIR, "power264CommunityAffiliation.1D")
POWER_COMM_NAMES = os.path.join(ATLAS_DIR, "power264CommunityNames.txt")
YEO_NII     = os.path.join(ATLAS_DIR, "Yeo2011_7Networks_MNI152_FreeSurferConformed1mm.nii")

OUT_DIR = os.path.join(BASE_DIR, "possible_edge_outputs")
os.makedirs(OUT_DIR, exist_ok=True)
OUT_XLSX = os.path.join(OUT_DIR, "Barrett_2020_possible_edges_master.xlsx")

YEO7 = ["Visual","Somatomotor","DorsalAttention","VentralAttention","Limbic","Frontoparietal","Default"]

def clean_ws(s):
    return re.sub(r"\s+", " ", str(s).strip())

def norm(s):
    s = clean_ws(s).lower()
    return re.sub(r"[^a-z0-9]+", "", s)

def dice(a, b):
    inter = np.count_nonzero(a & b)
    na = np.count_nonzero(a)
    nb = np.count_nonzero(b)
    return 0.0 if (na + nb) == 0 else (2.0 * inter) / (na + nb)

def possible_edges_for_pair(a, b, yeo_counts):
    nA = int(yeo_counts.get(a, 0))
    nB = int(yeo_counts.get(b, 0))
    if a == b:
        return nA, nB, nA * (nA - 1) // 2, "within"
    return nA, nB, nA * nB, "between"

def load_lines(path):
    out = []
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for ln in f:
            ln = clean_ws(ln)
            if ln:
                out.append(ln)
    return out

node_names = load_lines(POWER_NAMES)
community_names = load_lines(POWER_COMM_NAMES)

community_affil = []
with open(POWER_COMM_AFFIL, "r", encoding="utf-8", errors="ignore") as f:
    for ln in f:
        ln = clean_ws(ln)
        if not ln:
            continue
        community_affil.extend([int(v) for v in re.findall(r"-?\d+", ln)])

if len(node_names) != len(community_affil):
    raise ValueError(
        f"Power node names ({len(node_names)}) and community affiliations ({len(community_affil)}) differ."
    )

min_aff, max_aff = min(community_affil), max(community_affil)
if min_aff == 0 and max_aff == len(community_names) - 1:
    aff_to_name = {i: community_names[i] for i in range(len(community_names))}
elif min_aff == 1 and max_aff == len(community_names):
    aff_to_name = {i: community_names[i - 1] for i in range(1, len(community_names) + 1)}
else:
    raise ValueError(
        f"Could not align community IDs ({min_aff}..{max_aff}) with community names (n={len(community_names)})."
    )

node_df = pd.DataFrame({
    "Power_Node_ID": np.arange(1, len(node_names) + 1),
    "Power_Node_Name": node_names,
    "Power_Community_ID": community_affil,
})
node_df["Power_Community_Name"] = node_df["Power_Community_ID"].map(aff_to_name)

power_img = nib.load(POWER_NII)
yeo_img = nib.load(YEO_NII)

yd = yeo_img.get_fdata()
if yd.ndim == 4 and yd.shape[-1] == 1:
    yeo_img = nib.Nifti1Image(np.squeeze(yd, axis=-1), affine=yeo_img.affine)

yeo_rs = resample_to_img(yeo_img, power_img, interpolation="nearest")
power = power_img.get_fdata().astype(int)
yeo = yeo_rs.get_fdata().astype(int)

power_ids = [i for i in np.unique(power) if i != 0]
yeo_ids = sorted([i for i in np.unique(yeo) if i != 0])[:7]
yeo_id_to_name = {yeo_ids[i]: YEO7[i] for i in range(min(7, len(yeo_ids)))}

power_to_yeo = {}
power_to_dice = {}
for pid in power_ids:
    pm = (power == pid)
    best_name, best_d, best_inter = None, -1.0, -1
    for yid in yeo_ids:
        ym = (yeo == yid)
        inter = np.count_nonzero(pm & ym)
        d = dice(pm, ym)
        if (d > best_d) or (np.isclose(d, best_d) and inter > best_inter):
            best_d, best_inter = float(d), int(inter)
            best_name = yeo_id_to_name.get(yid)
    power_to_yeo[pid] = best_name
    power_to_dice[pid] = best_d

node_df["Yeo7"] = node_df["Power_Node_ID"].map(power_to_yeo)
node_df["Dice"] = node_df["Power_Node_ID"].map(power_to_dice)

yeo_counts = (
    node_df.groupby("Yeo7", as_index=False)["Power_Node_ID"]
    .nunique()
    .rename(columns={"Power_Node_ID": "N_Power_Nodes"})
)
yeo_counts = yeo_counts.set_index("Yeo7")["N_Power_Nodes"].to_dict()
yeo_counts = {k: int(yeo_counts.get(k, 0)) for k in YEO7}

yeo_counts_df = pd.DataFrame({
    "Yeo7": YEO7,
    "N_Power_Nodes": [yeo_counts[k] for k in YEO7]
})

node_name_lookup = {norm(x): x for x in node_df["Power_Node_Name"].astype(str)}
community_name_lookup = {norm(x): x for x in node_df["Power_Community_Name"].dropna().astype(str).unique()}

COMMUNITY_ALIAS = {
    "visual": "Visual",
    "dorsalattention": "Dorsal Attention",
    "defaultmode": "Default Mode",
    "defaultmodenetwork": "Default Mode",
    "dorsalssm": "Dorsal SSM",
    "ventralssm": "Ventral SSM",
    "auditory": "Auditory",
    "cotaskcontrol": "C-O Task Control",
    "cotask": "C-O Task Control",
}

def _laterality_candidates(raw):
    """
    Try to find Power node names corresponding to labels like Left Insula / Right Insula.
    """
    raw_clean = clean_ws(raw)
    raw_norm = norm(raw_clean)

    # exact first
    if raw_norm in node_name_lookup:
        return [node_name_lookup[raw_norm]]

    side = None
    if raw_norm.startswith("left"):
        side = "left"
        region = raw_norm[len("left"):]
    elif raw_norm.startswith("right"):
        side = "right"
        region = raw_norm[len("right"):]
    else:
        region = raw_norm

    candidates = []
    for lab in node_df["Power_Node_Name"].astype(str):
        lab_norm = norm(lab)

        if region and region not in lab_norm:
            continue

        if side == "left":
            if any(tok in lab_norm for tok in ["left", "lh", "_l", "linsula", "lins"]):
                candidates.append(lab)
        elif side == "right":
            if any(tok in lab_norm for tok in ["right", "rh", "_r", "rinsula", "rins"]):
                candidates.append(lab)
        else:
            candidates.append(lab)

    # fallback: any label containing the region string
    if not candidates and region:
        candidates = [lab for lab in node_df["Power_Node_Name"].astype(str) if region in norm(lab)]

    return list(dict.fromkeys(candidates))

def canonical_node_label(x, row_idx, colname):
    raw = clean_ws(x)
    candidates = _laterality_candidates(raw)

    if len(candidates) == 1:
        return candidates[0]

    if len(candidates) > 1:
        # choose the candidate with the highest Dice to the insula-like cortical node if possible is unsafe;
        # instead force manual inspection by raising with candidate list
        raise ValueError(
            f"[Barrett] Row {row_idx}: '{colname}' label {raw!r} matched multiple Power nodes.\n"
            f"Candidates:\n  - " + "\n  - ".join(candidates[:20])
        )

    hints = [lab for lab in node_df["Power_Node_Name"].astype(str)
             if norm(raw) in norm(lab) or norm(lab) in norm(raw)][:15]
    hint_txt = "\nClosest node matches:\n  - " + "\n  - ".join(hints) if hints else ""
    raise ValueError(
        f"[Barrett] Row {row_idx}: could not map '{colname}' label to Power node.\n"
        f"  Raw value: {raw!r}{hint_txt}"
    )

def canonical_comm_label(x, row_idx, colname):
    raw = clean_ws(x)
    k_norm = norm(raw)
    canon = COMMUNITY_ALIAS.get(k_norm, raw)
    canon_norm = norm(canon)

    if canon_norm in community_name_lookup:
        return community_name_lookup[canon_norm]

    hints = [lab for lab in community_name_lookup.values()
             if canon_norm in norm(lab) or norm(lab) in canon_norm][:10]
    hint_txt = "\nClosest community matches:\n  - " + "\n  - ".join(hints) if hints else ""
    raise ValueError(
        f"[Barrett] Row {row_idx}: could not map '{colname}' label to Power community.\n"
        f"  Raw value: {raw!r}\n"
        f"  Canonical attempted: {canon!r}{hint_txt}"
    )

master_rows = []
for i, a in enumerate(YEO7):
    for b in YEO7[i:]:
        nA, nB, possible, pair_type = possible_edges_for_pair(a, b, yeo_counts)
        master_rows.append({
            "Study": "Barrett 2020",
            "Atlas": "Power 264",
            "Yeo_Pair_A": a,
            "Yeo_Pair_B": b,
            "Regions_in_A": nA,
            "Regions_in_B": nB,
            "Possible_Edges": possible,
            "Pair_Type": pair_type,
        })
master_df = pd.DataFrame(master_rows)

df = pd.read_excel(BARRETT_XLSX)
df.columns = df.columns.astype(str).str.replace("\u00a0", " ", regex=False).str.strip()

required_cols = {"Study","Contrast","Seed","Target","Direction","Network"}
missing = required_cols - set(df.columns)
if missing:
    raise ValueError(f"Barrett file missing required columns: {sorted(missing)}")

net_norm = (
    df["Network"].astype(str).str.strip().str.lower()
    .str.replace(r"[^a-z0-9]+", "", regex=True)
)
sub = df[net_norm.isin(["poweratlas", "power264", "power"])].copy()
if len(sub) == 0:
    sub = df.copy()

comm_to_yeo = (
    node_df.groupby(["Power_Community_Name", "Yeo7"]).size().reset_index(name="n")
    .sort_values(["Power_Community_Name", "n", "Yeo7"], ascending=[True, False, True])
    .drop_duplicates("Power_Community_Name")
    .set_index("Power_Community_Name")["Yeo7"]
    .to_dict()
)

seed_node_exact = []
target_comm_exact = []
seed_node_id = []
seed_comm = []
seed_yeo = []
target_yeo = []
pair_a = []
pair_b = []
possible_edges = []
pair_type_list = []

for i, r in sub.iterrows():
    s_node = canonical_node_label(r["Seed"], i, "Seed")
    t_comm = canonical_comm_label(r["Target"], i, "Target")

    s_row = node_df[node_df["Power_Node_Name"] == s_node]
    if len(s_row) != 1:
        raise ValueError(f"[Barrett] Row {i}: seed node {s_node!r} matched {len(s_row)} Power nodes.")
    s_row = s_row.iloc[0]

    s_pid = int(s_row["Power_Node_ID"])
    s_comm = str(s_row["Power_Community_Name"])
    s_yeo = str(s_row["Yeo7"])
    t_yeo = comm_to_yeo.get(t_comm)

    if s_yeo not in YEO7:
        raise ValueError(f"[Barrett] Row {i}: seed node {s_node!r} mapped to invalid Yeo label {s_yeo!r}")
    if t_yeo not in YEO7:
        raise ValueError(f"[Barrett] Row {i}: target community {t_comm!r} mapped to invalid Yeo label {t_yeo!r}")

    A, B = (s_yeo, t_yeo) if YEO7.index(s_yeo) <= YEO7.index(t_yeo) else (t_yeo, s_yeo)
    _, _, denom, ptype = possible_edges_for_pair(A, B, yeo_counts)

    seed_node_exact.append(s_node)
    target_comm_exact.append(t_comm)
    seed_node_id.append(s_pid)
    seed_comm.append(s_comm)
    seed_yeo.append(s_yeo)
    target_yeo.append(t_yeo)
    pair_a.append(A)
    pair_b.append(B)
    possible_edges.append(denom)
    pair_type_list.append(ptype)

rowlevel_df = pd.DataFrame({
    "Study": sub["Study"].astype(str).values,
    "Contrast": sub["Contrast"].astype(str).values,
    "Seed_Original": sub["Seed"].astype(str).values,
    "Target_Original": sub["Target"].astype(str).values,
    "Seed_Power_Node": seed_node_exact,
    "Seed_Power_Node_ID": seed_node_id,
    "Seed_Power_Community": seed_comm,
    "Target_Power_Community": target_comm_exact,
    "Seed_Yeo7": seed_yeo,
    "Target_Yeo7": target_yeo,
    "Yeo_Pair_A": pair_a,
    "Yeo_Pair_B": pair_b,
    "Direction": sub["Direction"].astype(str).values,
    "Possible_Edges": possible_edges,
    "Pair_Type": pair_type_list,
})

with pd.ExcelWriter(OUT_XLSX, engine="openpyxl") as writer:
    node_df.to_excel(writer, sheet_name="Power_to_Yeo7_mapping", index=False)
    yeo_counts_df.to_excel(writer, sheet_name="Yeo_counts", index=False)
    master_df.to_excel(writer, sheet_name="Master_possible_edges", index=False)
    rowlevel_df.to_excel(writer, sheet_name="Rowlevel_denominators", index=False)

print("✅ Saved:", OUT_XLSX)
print("\nIf this still errors, it will now show candidate Power node names for manual aliasing.")

ValueError: [Barrett] Row 0: could not map 'Seed' label to Power node.
  Raw value: 'Left Insula'